# ATRA-4B — does the *served* q4_k_m GGUF meet the thresholds?

The continuation run `atra12/atra-4b-continuation-20260921` added 400 steps
to the original adapter with the trainer bugs fixed, and `evaluate.py`
returned exit code 0 on it — against the **adapter, through transformers**.
That run recorded `deployment_validated: false` in its own result, because
what anyone downloads and runs is not the adapter: it is the q4_k_m GGUF
served by `llama-server`.

That path has produced two wrong answers in this project already:

1. the exported GGUF carried **no chat template**, so llama-server fell back
   to a built-in default with no `tools` block and every training prompt
   shape was lost — prompts arrived at 91 tokens;
2. sending `tools` switched on llama.cpp's tool-call parser, which moved the
   model's reply out of `message.content`, and the scorer read almost
   nothing.

Both are fixed in `train.py` (writes `chat_template.jinja` beside the
weights), `export.py` (recovers a template, refuses to export without one,
re-reads the merged directory to prove it survived) and `evaluate.py` (sends
the tools, reads `tool_calls`, and counts generated tokens against the
tokens that reached the scorer). This kernel tests whether they are fixed
**in the artifact**.

Nothing here is tuned. The thresholds come from the repository's own
`config/default.yaml`, carried byte-for-byte and digest-checked; the test
split is rebuilt with the exact command the thresholds were written for; the
exit code is reported as it comes. **A failure is the useful outcome if it
is the true one** — it would mean the served path still differs from the
trained one, which is precisely what this run exists to detect.

## What the run does

1. pinned dependencies, minus torchao;
2. the **new** adapter staged from the continuation kernel (nothing is
   retrained), with its `chat_template.jinja`;
3. merged into `Qwen/Qwen3-4B-Instruct-2507` and converted to q4_k_m;
4. the template read back **out of the GGUF** and rendered, with and without
   tools, before anything is served;
5. `llama-server` given the GGUF and `--jinja` and *nothing else* — no
   `--chat-template-file`, because patching the template on the command line
   would fix the measurement and not the artifact;
6. the test split rebuilt with
   `python -m data.build --seed 42 --per-domain 200 --out data/out`;
7. `evaluate.py` run twice against that one server: once on
   `/v1/chat/completions` with the tools (how the runtime and Ollama talk to
   it) and once on `/completion` with the prompt rendered by
   `prompting.render_example` (how training rendered it). Same weights, same
   split, same thresholds; the two differ only in who renders and who
   parses.

In [ ]:
!nvidia-smi
import platform, subprocess, torch
print("torch", torch.__version__, "cuda", torch.version.cuda)
print("python", platform.python_version())
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True).stdout)
!ls -la /kaggle/input/*
!df -h

## 1. Dependencies

The training run's pins, minus torchao.

In [ ]:
%%bash
set -e
set -o pipefail
pip install -q "transformers==4.57.6" "peft==0.19.1" "accelerate==1.12.0" \
  "huggingface_hub==0.36.2" "sentencepiece==0.2.1" "pyyaml==6.0.3"
# Kaggle ships torchao 0.10.0. peft's LoRA dispatcher calls
# is_torchao_available() for every module it injects, and that helper raises
# on a version below 0.16.0 rather than returning False. The merge loads the
# base in fp16, so it reaches that dispatcher. Neither path uses torchao.
pip uninstall -q -y torchao || true
python - <<'PY'
import importlib.util
print("torchao present:", importlib.util.find_spec("torchao") is not None)
import accelerate, jinja2, peft, transformers, yaml
print("deps ok:", transformers.__version__, peft.__version__, accelerate.__version__)
print("jinja2 :", jinja2.__version__)
PY

## 2. The repository files

`evaluate.py`, `prompting.py`, `export.py`, the `data` package and
`config/default.yaml`, carried verbatim rather than cloned, so the run
cannot silently score against a different revision than the one the result
is reported for. Each digest is compared in-kernel against the working tree
as it stood when this notebook was generated.

In [ ]:
import base64, hashlib, pathlib, shutil

# The ATRA repository files this evaluation needs, carried verbatim as base64.
# Carried rather than cloned so the run cannot silently evaluate a different
# revision of evaluate.py, prompting.py, data/build.py or config/default.yaml
# than the one whose thresholds this result will be reported against.
#
# The blobs are not hand-written: `make-notebook.js` reads each file from the
# working tree at generation time, so refreshing the notebook refreshes the
# harness. EXPECTED holds the digest of the file as it stood in the working
# tree when this notebook was generated, and the assertion below makes the
# comparison part of the run rather than a promise about it.
FILES = {
    "evaluate.py": (
        "..."
        "dChtYWluKCkpDQo="
    ),
    "prompting.py": (
        "IiIiVGhlIG9uZSBwbGFjZSBhIHByb21wdCBpcyByZW5kZXJlZC4KClR3byBldmFsdWF0aW9u"
        "cyBoYXZlIG5vdyBiZWVuIHRocm93biBhd2F5IGJlY2F1c2UgdHJhaW5pbmcgYW5kIGV2YWx1"
        "YXRpb24gZGlkCm5vdCByZW5kZXIgdGhlIHNhbWUgcHJvbXB0OgoKKiBydW4gMSBzY29yZWQg"
        "dGhlIG1vZGVsIG9uIGBtZXNzYWdlc2AgYWxvbmUgd2hpbGUgZXZlcnkgdHJhaW5pbmcgcHJv"
        "bXB0IGhhZAogIGJlZW4gcmVuZGVyZWQgd2l0aCBgYXBwbHlfY2hhdF90ZW1wbGF0ZShtZXNz"
        "YWdlcywgdG9vbHM9Li4uKWAsIHNvIHRoZSB0b29sCiAgYmxvY2sg4oCUIHNldmVyYWwgaHVu"
        "ZHJlZCB0b2tlbnMgb2Ygc3lzdGVtIHRleHQg4oCUIHdhcyBtaXNzaW5nIGF0IGV2YWwgdGlt"
        "ZTsKKiBydW4gMiBzZW50IHRoZSB0b29scywgYW5kIHRoZSBudW1iZXJzIHdlcmUgc3RpbGwg"
        "dW51c2FibGUsIGJlY2F1c2Ugc2VuZGluZwogIGB0b29sc2AgdG8gYC92MS9jaGF0L2NvbXBs"
        "ZXRpb25zYCBzd2l0Y2hlcyBvbiBsbGFtYS5jcHAncyB0b29sLWNhbGwgcGFyc2VyLAogIHdo"
        "aWNoIHJld3JvdGUgdGhlIHJlcGx5IGJlZm9yZSB0aGUgc2NvcmVyIGV2ZXIgc2F3IGl0LgoK"
        "Qm90aCBhcmUgdGhlIHNhbWUgY2xhc3Mgb2YgYnVnOiB0d28gY29kZSBwYXRocyB0aGF0IHdl"
        "cmUgKnN1cHBvc2VkKiB0byBhZ3JlZS4KU28gdGhlcmUgaXMgbm93IGV4YWN0bHkgb25lIGZ1"
        "bmN0aW9uIHRoYXQgdHVybnMgYW4gYEV4YW1wbGVgIGludG8gdGV4dCwgaXQKbGl2ZXMgaGVy"
        "ZSwgYW5kIGB0cmFpbi5weWAgYW5kIGBldmFsdWF0ZS5weWAgYm90aCBjYWxsIGl0LiBgY29t"
        "cGxldGlvbl9wYWlyYApzcGxpdHMgdGhhdCBzaW5nbGUgcmVuZGVyaW5nIGF0IHRoZSBnZW5l"
        "cmF0aW9uIGJvdW5kYXJ5LCB3aGljaCBpcyB3aGF0IG1ha2VzCiJ0aGUgcHJvbXB0IHRoZSBt"
        "b2RlbCBpcyBzY29yZWQgb24iIGFuZCAidGhlIHByb21wdCB0aGUgbG9zcyB3YXMgbWFza2Vk"
        "IHRvIgp0aGUgc2FtZSBzdHJpbmcgYnkgY29uc3RydWN0aW9uIHJhdGhlciB0aGFuIGJ5IHJl"
        "dmlldy4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIHR5"
        "cGluZyBpbXBvcnQgQW55Cgpmcm9tIGRhdGEuc2NoZW1hIGltcG9ydCBFeGFtcGxlCgoKZGVm"
        "IHRvb2xfc2NoZW1hcyhleGFtcGxlOiBFeGFtcGxlKSAtPiBsaXN0W2RpY3Rbc3RyLCBBbnld"
        "XToKICAgICIiIlRoZSBleGFtcGxlJ3MgdG9vbHMgaW4gdGhlIE9wZW5BSSBmdW5jdGlvbiBz"
        "aGFwZS4KCiAgICBUaGUgc2FtZSBzaGFwZSBnb2VzIHRvIGBhcHBseV9jaGF0X3RlbXBsYXRl"
        "KHRvb2xzPS4uLilgIGFuZCBpbnRvIGFuCiAgICBPcGVuQUktY29tcGF0aWJsZSByZXF1ZXN0"
        "IGJvZHksIHNvIGEgc2VydmVyIHRoYXQgcmVuZGVycyB0aGUgdGVtcGxhdGUKICAgIGl0c2Vs"
        "ZiBhbmQgdGhpcyBtb2R1bGUgY2Fubm90IGRpc2FncmVlIGFib3V0IHdoYXQgdGhlIHRvb2xz"
        "IGFyZS4KICAgICIiIgogICAgcmV0dXJuIFsKICAgICAgICB7CiAgICAgICAgICAgICJ0eXBl"
        "IjogImZ1bmN0aW9uIiwKICAgICAgICAgICAgImZ1bmN0aW9uIjogewogICAgICAgICAgICAg"
        "ICAgIm5hbWUiOiB0b29sLm5hbWUsCiAgICAgICAgICAgICAgICAiZGVzY3JpcHRpb24iOiB0"
        "b29sLmRlc2NyaXB0aW9uLAogICAgICAgICAgICAgICAgInBhcmFtZXRlcnMiOiB0b29sLnBh"
        "cmFtZXRlcnMsCiAgICAgICAgICAgIH0sCiAgICAgICAgfQogICAgICAgIGZvciB0b29sIGlu"
        "IGV4YW1wbGUudG9vbHMKICAgIF0KCgpkZWYgcmVuZGVyX2V4YW1wbGUoZXhhbXBsZTogRXhh"
        "bXBsZSwgdG9rZW5pemVyOiBBbnksICosIHByb21wdF9vbmx5OiBib29sID0gRmFsc2UpIC0+"
        "IHN0cjoKICAgICIiIlJlbmRlciBvbmUgZXhhbXBsZSB0aHJvdWdoIHRoZSBtb2RlbCdzIG93"
        "biBjaGF0IHRlbXBsYXRlLgoKICAgIFVzaW5nIHRoZSB0b2tlbml6ZXIncyB0ZW1wbGF0ZSBy"
        "YXRoZXIgdGhhbiBhIGhhbmQtcm9sbGVkIGZvcm1hdCBtYXR0ZXJzOiB0aGUKICAgIHNwZWNp"
        "YWwgdG9rZW5zIGEgbW9kZWwgd2FzIHByZXRyYWluZWQgd2l0aCBhcmUgcGFydCBvZiBpdHMg"
        "aW50ZXJmYWNlLCBhbmQKICAgIGdldHRpbmcgdGhlbSB3cm9uZyBwcm9kdWNlcyBhIG1vZGVs"
        "IHRoYXQgd29ya3MgaW4gZXZhbHVhdGlvbiBhbmQgZmFpbHMgaW4KICAgIHRoZSBydW50aW1l"
        "LgoKICAgIGBwcm9tcHRfb25seWAgZHJvcHMgdGhlIGFzc2lzdGFudCB0dXJuIGFuZCBhcHBl"
        "bmRzIHRoZSBnZW5lcmF0aW9uIHByb21wdCDigJQKICAgIHRoZSBleGFjdCBwcmVmaXggdGhl"
        "IG1vZGVsIGlzIGFza2VkIHRvIGNvbnRpbnVlIGF0IGluZmVyZW5jZSB0aW1lLgogICAgIiIi"
        "CiAgICBtZXNzYWdlcyA9IFsKICAgICAgICB7InJvbGUiOiBtZXNzYWdlLnJvbGUsICJjb250"
        "ZW50IjogbWVzc2FnZS5jb250ZW50fSBmb3IgbWVzc2FnZSBpbiBleGFtcGxlLm1lc3NhZ2Vz"
        "CiAgICBdCiAgICBpZiBwcm9tcHRfb25seToKICAgICAgICBpZiBub3QgbWVzc2FnZXMgb3Ig"
        "bWVzc2FnZXNbLTFdWyJyb2xlIl0gIT0gImFzc2lzdGFudCI6CiAgICAgICAgICAgIHJhaXNl"
        "IFZhbHVlRXJyb3IoInRyYWluaW5nIGV4YW1wbGUgbXVzdCBlbmQgaW4gYW4gYXNzaXN0YW50"
        "IGFuc3dlciIpCiAgICAgICAgbWVzc2FnZXMgPSBtZXNzYWdlc1s6LTFdCgogICAgdG9vbHMg"
        "PSB0b29sX3NjaGVtYXMoZXhhbXBsZSkKCiAgICB0cnk6CiAgICAgICAgcmV0dXJuIHRva2Vu"
        "aXplci5hcHBseV9jaGF0X3RlbXBsYXRlKAogICAgICAgICAgICBtZXNzYWdlcywgdG9vbHM9"
        "dG9vbHMgb3IgTm9uZSwgdG9rZW5pemU9RmFsc2UsIGFkZF9nZW5lcmF0aW9uX3Byb21wdD1w"
        "cm9tcHRfb25seQogICAgICAgICkKICAgIGV4Y2VwdCBUeXBlRXJyb3I6CiAgICAgICAgIyBP"
        "bGRlciB0ZW1wbGF0ZXMgZG8gbm90IGFjY2VwdCBgdG9vbHNgLgogICAgICAgIHJldHVybiB0"
        "b2tlbml6ZXIuYXBwbHlfY2hhdF90ZW1wbGF0ZSgKICAgICAgICAgICAgbWVzc2FnZXMsIHRv"
        "a2VuaXplPUZhbHNlLCBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQ9cHJvbXB0X29ubHkKICAgICAg"
        "ICApCgoKZGVmIGNvbXBsZXRpb25fcGFpcihleGFtcGxlOiBFeGFtcGxlLCB0b2tlbml6ZXI6"
        "IEFueSkgLT4gZGljdFtzdHIsIHN0cl06CiAgICAiIiJLZWVwIHRoZSBpbmZlcmVuY2UgcHJl"
        "Zml4IGlkZW50aWNhbCB3aGlsZSBtYXNraW5nIGl0IG91dCBvZiB0aGUgbG9zcy4iIiIKICAg"
        "IHByb21wdCA9IHJlbmRlcl9leGFtcGxlKGV4YW1wbGUsIHRva2VuaXplciwgcHJvbXB0X29u"
        "bHk9VHJ1ZSkKICAgIGZ1bGwgPSByZW5kZXJfZXhhbXBsZShleGFtcGxlLCB0b2tlbml6ZXIp"
        "CiAgICBpZiBub3QgZnVsbC5zdGFydHN3aXRoKHByb21wdCk6CiAgICAgICAgcmFpc2UgVmFs"
        "dWVFcnJvcihmIntleGFtcGxlLmlkfTogdHJhaW5pbmcgYW5kIGdlbmVyYXRpb24gcHJlZml4"
        "ZXMgZGlmZmVyIikKICAgIGNvbXBsZXRpb24gPSBmdWxsW2xlbihwcm9tcHQpIDpdCiAgICBp"
        "ZiBub3QgY29tcGxldGlvbi5zdHJpcCgpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ7"
        "ZXhhbXBsZS5pZH06IGVtcHR5IGNvbXBsZXRpb24iKQogICAgcmV0dXJuIHsicHJvbXB0Ijog"
        "cHJvbXB0LCAiY29tcGxldGlvbiI6IGNvbXBsZXRpb259Cg=="
    ),
    "export.py": (
        "IiIiTWVyZ2UgYSB0cmFpbmVkIEFUUkEtNEIgTG9SQSBhZGFwdGVyIGludG8gaXRzIGJhc2Ug"
        "bW9kZWwgYW5kIGV4cG9ydCBpdC4KClR3byBvdXRwdXRzLCBib3RoIG9wdGlvbmFsOgoKMS4g"
        "YGA8b3V0Pi9tZXJnZWRgYCDigJQgdGhlIGJhc2UgbW9kZWwgd2l0aCB0aGUgYWRhcHRlciBm"
        "b2xkZWQgaW4sIHNhdmVkIGFzCiAgIHNhZmV0ZW5zb3JzIGluIGZwMTYgKG9yIGJmMTYgb24g"
        "aGFyZHdhcmUgdGhhdCBzdXBwb3J0cyBpdCkuIFRoaXMgaXMgd2hhdCBhCiAgIEdHVUYgY29u"
        "dmVydGVyIGNvbnN1bWVzLgoyLiBgYDxvdXQ+L2F0cmEtNGItPHR5cGU+LmdndWZgYCDigJQg"
        "YSBsbGFtYS5jcHAgR0dVRiBmaWxlLCBwcm9kdWNlZCBieSB0aGUKICAgYGBjb252ZXJ0X2hm"
        "X3RvX2dndWYucHlgYCBzY3JpcHQgb2YgYSBsbGFtYS5jcHAgY2hlY2tvdXQgdGhhdCB0aGUg"
        "Y2FsbGVyCiAgIHBvaW50cyBhdCB3aXRoIGBgLS1sbGFtYS1jcHBgYC4gYGBmMTZgYCBhbmQg"
        "YGBxOF8wYGAgbmVlZCBvbmx5IHRoYXQgUHl0aG9uCiAgIHNjcmlwdDsgYGBxNF9rX21gYCBh"
        "ZGRpdGlvbmFsbHkgbmVlZHMgYSBidWlsdCBgYGxsYW1hLXF1YW50aXplYGAgYmluYXJ5LAog"
        "ICB3aGljaCBpcyBsb29rZWQgZm9yIG5leHQgdG8gdGhlIGNoZWNrb3V0J3MgYGBidWlsZC9i"
        "aW5gYC4KCk5vdGhpbmcgaGVyZSB0cmFpbnMsIGV2YWx1YXRlcyBvciB1cGxvYWRzLiBJdCB3"
        "cml0ZXMgYGBleHBvcnQtbWFuaWZlc3QuanNvbmBgCm5leHQgdG8gdGhlIG91dHB1dHMgc28g"
        "YSBHR1VGIGNhbiBhbHdheXMgYmUgdHJhY2VkIGJhY2sgdG8gdGhlIGFkYXB0ZXIsIHRoZQpi"
        "YXNlIHJldmlzaW9uIGFuZCB0aGUgdHJhaW5pbmcgbWFuaWZlc3QgdGhhdCBwcm9kdWNlZCBp"
        "dCDigJQgYSBmaWxlIHRoYXQgbGFja3MKdGhhdCBjaGFpbiBpcyBub3QgYW4gQVRSQS00QiBy"
        "ZWxlYXNlLCB3aGF0ZXZlciBpdCBpcyBjYWxsZWQuCgpVc2FnZTo6CgogICAgcHl0aG9uIGV4"
        "cG9ydC5weSAtLWFkYXB0ZXIgcnVucy9hdHJhLTRiIC0tb3V0IGV4cG9ydAogICAgcHl0aG9u"
        "IGV4cG9ydC5weSAtLWFkYXB0ZXIgcnVucy9hdHJhLTRiIC0tb3V0IGV4cG9ydCBcCiAgICAg"
        "ICAgLS1sbGFtYS1jcHAgL2thZ2dsZS93b3JraW5nL2xsYW1hLmNwcCAtLWdndWYgcThfMAoK"
        "VGhlIGFkYXB0ZXIgZGlyZWN0b3J5IG11c3QgY29udGFpbiBgYG1hbmlmZXN0Lmpzb25gYCBm"
        "cm9tIGBgdHJhaW4ucHlgYCB3aXRoCmBgc3RhdHVzOiBjb21wbGV0ZWRgYC4gQSBzbW9rZSBt"
        "YW5pZmVzdCBpcyByZWZ1c2VkOiBhIDMwLXN0ZXAgY2hlY2twb2ludCBpcyBub3QKYSBtb2Rl"
        "bCwgYW5kIGV4cG9ydGluZyBpdCB3b3VsZCBvbmx5IG1ha2UgaXQgbG9vayBsaWtlIG9uZS4K"
        "IiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFy"
        "c2UKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBzaHV0aWwK"
        "aW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIg"
        "aW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKIyBUaGUgZnVsbC1wcmVjaXNp"
        "b24gYmFzZSB0aGUgYWRhcHRlciBpcyBtZXJnZWQgaW50by4gVGhlIGFkYXB0ZXIgd2FzIHRy"
        "YWluZWQKIyBhZ2FpbnN0IHRoZSBibmItNGJpdCB2YXJpYW50IG9mIHRoaXMgZXhhY3QgY2hl"
        "Y2twb2ludDsgbWVyZ2luZyBpbnRvIHRoZQojIGZwMTYgd2VpZ2h0cyBpcyB0aGUgc3RhbmRh"
        "cmQgUUxvUkEgZXhwb3J0IHBhdGggYW5kIHRoZSBtYW5pZmVzdCByZWNvcmRzIGJvdGguCkRF"
        "RkFVTFRfRlVMTF9CQVNFID0gIlF3ZW4vUXdlbjMtNEItSW5zdHJ1Y3QtMjUwNyIKCk1PREVM"
        "RklMRSA9ICIiIiMgT2xsYW1hIE1vZGVsZmlsZSBmb3IgQVRSQS00Qi4KIwojIFRoZSBjaGF0"
        "IHRlbXBsYXRlIHRyYXZlbHMgaW5zaWRlIHRoZSBHR1VGICh0aGUgY29udmVydGVyIGVtYmVk"
        "cyB0aGUgYmFzZQojIG1vZGVsJ3MgdG9rZW5pemVyIGNoYXQgdGVtcGxhdGUpLCBzbyBub25l"
        "IGlzIHJlcGVhdGVkIGhlcmUuIFRoZSBBVFJBIHJ1bnRpbWUKIyBzdXBwbGllcyBpdHMgb3du"
        "IHN5c3RlbSBwcm9tcHQgcGVyIGFnZW50OyBkbyBub3QgYWRkIG9uZS4KRlJPTSAuL3tnZ3Vm"
        "fQpQQVJBTUVURVIgdGVtcGVyYXR1cmUgMC4xClBBUkFNRVRFUiBudW1fY3R4IHtjdHh9CiIi"
        "IgoKCmRlZiBzaGEyNTZfZmlsZShwYXRoOiBQYXRoKSAtPiBzdHI6CiAgICBkaWdlc3QgPSBo"
        "YXNobGliLnNoYTI1NigpCiAgICB3aXRoIHBhdGgub3BlbigicmIiKSBhcyBoYW5kbGU6CiAg"
        "ICAgICAgZm9yIGNodW5rIGluIGl0ZXIobGFtYmRhOiBoYW5kbGUucmVhZCgxIDw8IDIwKSwg"
        "YiIiKToKICAgICAgICAgICAgZGlnZXN0LnVwZGF0ZShjaHVuaykKICAgIHJldHVybiBkaWdl"
        "c3QuaGV4ZGlnZXN0KCkKCgpkZWYgbG9hZF90cmFpbmluZ19tYW5pZmVzdChhZGFwdGVyOiBQ"
        "YXRoKSAtPiBkaWN0W3N0ciwgQW55XToKICAgIG1hbmlmZXN0X3BhdGggPSBhZGFwdGVyIC8g"
        "Im1hbmlmZXN0Lmpzb24iCiAgICBpZiBub3QgbWFuaWZlc3RfcGF0aC5leGlzdHMoKToKICAg"
        "ICAgICByYWlzZSBTeXN0ZW1FeGl0KGYie21hbmlmZXN0X3BhdGh9IGlzIG1pc3Npbmc7IGV4"
        "cG9ydCBvbmx5IHdoYXQgdHJhaW4ucHkgcHJvZHVjZWQiKQogICAgbWFuaWZlc3QgPSBqc29u"
        "LmxvYWRzKG1hbmlmZXN0X3BhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQogICAg"
        "c3RhdHVzID0gbWFuaWZlc3QuZ2V0KCJzdGF0dXMiKQogICAgaWYgc3RhdHVzICE9ICJjb21w"
        "bGV0ZWQiOgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoCiAgICAgICAgICAgIGYiYWRhcHRl"
        "ciBtYW5pZmVzdCBzdGF0dXMgaXMge3N0YXR1cyFyfSwgbm90ICdjb21wbGV0ZWQnOyAiCiAg"
        "ICAgICAgICAgICJhIHNtb2tlIG9yIGZhaWxlZCBydW4gbXVzdCBub3QgYmUgZXhwb3J0ZWQg"
        "YXMgYSBtb2RlbCIKICAgICAgICApCiAgICByZXR1cm4gbWFuaWZlc3QKCgpkZWYgbWVyZ2Uo"
        "YWRhcHRlcjogUGF0aCwgYmFzZTogc3RyLCBvdXQ6IFBhdGgpIC0+IFBhdGg6CiAgICB0cnk6"
        "CiAgICAgICAgaW1wb3J0IHRvcmNoCiAgICAgICAgZnJvbSBwZWZ0IGltcG9ydCBQZWZ0TW9k"
        "ZWwKICAgICAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgQXV0b01vZGVsRm9yQ2F1c2Fs"
        "TE0sIEF1dG9Ub2tlbml6ZXIKICAgIGV4Y2VwdCBJbXBvcnRFcnJvciBhcyBlcnJvcjogICMg"
        "cHJhZ21hOiBubyBjb3ZlciAtIGRlcGVuZHMgb24gdGhlIGVudmlyb25tZW50CiAgICAgICAg"
        "cmFpc2UgU3lzdGVtRXhpdChmIm1lcmdlIGRlcGVuZGVuY2llcyBhcmUgbm90IGluc3RhbGxl"
        "ZCAoe2Vycm9yfSkiKSBmcm9tIGVycm9yCgogICAgbWVyZ2VkX2RpciA9IG91dCAvICJtZXJn"
        "ZWQiCiAgICBpZiBtZXJnZWRfZGlyLmV4aXN0cygpOgogICAgICAgIHNodXRpbC5ybXRyZWUo"
        "bWVyZ2VkX2RpcikKICAgIG1lcmdlZF9kaXIubWtkaXIocGFyZW50cz1UcnVlKQoKICAgIHVz"
        "ZV9jdWRhID0gdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKQogICAgIyBOYXRpdmUgYmYxNiBv"
        "bmx5IChBbXBlcmUgYW5kIG5ld2VyKTsgc2VlIHN1cHBvcnRzX2JmMTYgaW4gdHJhaW4ucHkg"
        "Zm9yCiAgICAjIHdoeSB0b3JjaC5jdWRhLmlzX2JmMTZfc3VwcG9ydGVkKCkgaXMgdGhlIHdy"
        "b25nIHF1ZXN0aW9uIG9uIGEgVDQuCiAgICBjYXBhYmlsaXR5ID0gdG9yY2guY3VkYS5nZXRf"
        "ZGV2aWNlX2NhcGFiaWxpdHkoKSBpZiB1c2VfY3VkYSBlbHNlICgwLCAwKQogICAgZHR5cGUg"
        "PSB0b3JjaC5iZmxvYXQxNiBpZiBjYXBhYmlsaXR5WzBdID49IDggZWxzZSB0b3JjaC5mbG9h"
        "dDE2CiAgICBwcmludChmImJhc2UgbW9kZWwgOiB7YmFzZX0iKQogICAgcHJpbnQoZiJhZGFw"
        "dGVyICAgIDoge2FkYXB0ZXJ9IikKICAgIHByaW50KGYiZHR5cGUgICAgICA6IHtkdHlwZX0i"
        "KQoKICAgIG1vZGVsID0gQXV0b01vZGVsRm9yQ2F1c2FsTE0uZnJvbV9wcmV0cmFpbmVkKAog"
        "ICAgICAgIGJhc2UsIHRvcmNoX2R0eXBlPWR0eXBlLCBkZXZpY2VfbWFwPSJhdXRvIiBpZiB1"
        "c2VfY3VkYSBlbHNlIE5vbmUsIGxvd19jcHVfbWVtX3VzYWdlPVRydWUKICAgICkKICAgIG1v"
        "ZGVsID0gUGVmdE1vZGVsLmZyb21fcHJldHJhaW5lZChtb2RlbCwgc3RyKGFkYXB0ZXIpKQog"
        "ICAgbW9kZWwgPSBtb2RlbC5tZXJnZV9hbmRfdW5sb2FkKCkKICAgIG1vZGVsLnNhdmVfcHJl"
        "dHJhaW5lZChzdHIobWVyZ2VkX2RpciksIHNhZmVfc2VyaWFsaXphdGlvbj1UcnVlKQoKICAg"
        "ICMgVGhlIHRva2VuaXplciBjb21lcyBmcm9tIHRoZSBhZGFwdGVyLCBidXQgaXRzIGNoYXQg"
        "dGVtcGxhdGUgbWF5IG5vdC4KICAgICMgdHJhbnNmb3JtZXJzIDQuNTcgd3JpdGVzIHRoZSB0"
        "ZW1wbGF0ZSB0byBhIHNlcGFyYXRlIGNoYXRfdGVtcGxhdGUuamluamEKICAgICMgcmF0aGVy"
        "IHRoYW4gaW50byB0b2tlbml6ZXJfY29uZmlnLmpzb24sIGFuZCB0aGF0IGZpbGUgZG9lcyBu"
        "b3QgYWx3YXlzCiAgICAjIHRyYXZlbCB3aXRoIGFuIGFkYXB0ZXIgZGlyZWN0b3J5LiBBIHRv"
        "a2VuaXplciB3aXRoIG5vIHRlbXBsYXRlIHByb2R1Y2VzCiAgICAjIGEgR0dVRiB3aXRoIG5v"
        "IHRlbXBsYXRlLCBhbmQgbGxhbWEtc2VydmVyIHRoZW4gZmFsbHMgYmFjayB0byBhIGJ1aWx0"
        "LWluCiAgICAjIGRlZmF1bHQgdGhhdCBoYXMgbm8gdG9vbHMgYmxvY2sg4oCUIHNvIHRoZSBt"
        "b2RlbCBpcyBzZXJ2ZWQgYSBwcm9tcHQgc2hhcGUKICAgICMgaXQgd2FzIG5ldmVyIHRyYWlu"
        "ZWQgb24sIGFuZCBub3RoaW5nIGRvd25zdHJlYW0gc2F5cyBhIHdvcmQgYWJvdXQgaXQuCiAg"
        "ICAjIFRoYXQgaXMgaG93IHR3byBldmFsdWF0aW9uIHJ1bnMgd2VyZSBzY29yZWQgYWdhaW5z"
        "dCB0aGUgd3JvbmcgcHJvbXB0LgogICAgIwogICAgIyBUaGUgYmFzZSBtb2RlbCBpcyB3aGVy"
        "ZSB0aGUgdGVtcGxhdGUgY2FtZSBmcm9tIGluIHRoZSBmaXJzdCBwbGFjZTogaXQgaXMKICAg"
        "ICMgdGhlIHRva2VuaXplciB0cmFpbi5weSByZW5kZXJlZCBhbGwgODAwIHRyYWluaW5nIHBy"
        "b21wdHMgd2l0aC4KICAgIHRva2VuaXplciA9IEF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFp"
        "bmVkKHN0cihhZGFwdGVyKSkKICAgIGlmIG5vdCBnZXRhdHRyKHRva2VuaXplciwgImNoYXRf"
        "dGVtcGxhdGUiLCBOb25lKToKICAgICAgICBiYXNlX3Rva2VuaXplciA9IEF1dG9Ub2tlbml6"
        "ZXIuZnJvbV9wcmV0cmFpbmVkKGJhc2UpCiAgICAgICAgdGVtcGxhdGUgPSBnZXRhdHRyKGJh"
        "c2VfdG9rZW5pemVyLCAiY2hhdF90ZW1wbGF0ZSIsIE5vbmUpCiAgICAgICAgaWYgbm90IHRl"
        "bXBsYXRlOgogICAgICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KAogICAgICAgICAgICAgICAg"
        "ZiJuZWl0aGVyIHthZGFwdGVyfSBub3Ige2Jhc2V9IGNhcnJpZXMgYSBjaGF0IHRlbXBsYXRl"
        "OyByZWZ1c2luZyB0byIKICAgICAgICAgICAgICAgICIgZXhwb3J0IGEgbW9kZWwgdGhhdCBj"
        "YW5ub3QgYmUgc2VydmVkIHRoZSB3YXkgaXQgd2FzIHRyYWluZWQiCiAgICAgICAgICAgICkK"
        "ICAgICAgICB0b2tlbml6ZXIuY2hhdF90ZW1wbGF0ZSA9IHRlbXBsYXRlCiAgICAgICAgcHJp"
        "bnQoZiJ0ZW1wbGF0ZSAgIDogcmVjb3ZlcmVkIGZyb20ge2Jhc2V9ICh7bGVuKHRlbXBsYXRl"
        "KX0gY2hhcnMpIikKCiAgICB0b2tlbml6ZXIuc2F2ZV9wcmV0cmFpbmVkKHN0cihtZXJnZWRf"
        "ZGlyKSkKCiAgICAjIFByb3ZlIGl0IHN1cnZpdmVkIHRoZSByb3VuZCB0cmlwIHJhdGhlciB0"
        "aGFuIGFzc3VtaW5nIGl0IGRpZC4KICAgIHdyaXR0ZW4gPSBBdXRvVG9rZW5pemVyLmZyb21f"
        "cHJldHJhaW5lZChzdHIobWVyZ2VkX2RpcikpCiAgICBpZiBub3QgZ2V0YXR0cih3cml0dGVu"
        "LCAiY2hhdF90ZW1wbGF0ZSIsIE5vbmUpOgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoZiJ7"
        "bWVyZ2VkX2Rpcn0gd2FzIHdyaXR0ZW4gd2l0aG91dCBhIGNoYXQgdGVtcGxhdGUiKQoKICAg"
        "IHByaW50KGYibWVyZ2VkICAgICA6IHttZXJnZWRfZGlyfSIpCiAgICByZXR1cm4gbWVyZ2Vk"
        "X2RpcgoKCmRlZiBjb252ZXJ0X2dndWYobWVyZ2VkOiBQYXRoLCBvdXQ6IFBhdGgsIGxsYW1h"
        "X2NwcDogUGF0aCwgZ2d1Zl90eXBlOiBzdHIpIC0+IFBhdGg6CiAgICBjb252ZXJ0ZXIgPSBs"
        "bGFtYV9jcHAgLyAiY29udmVydF9oZl90b19nZ3VmLnB5IgogICAgaWYgbm90IGNvbnZlcnRl"
        "ci5leGlzdHMoKToKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KGYie2NvbnZlcnRlcn0gbm90"
        "IGZvdW5kOyBwYXNzIC0tbGxhbWEtY3BwIHBvaW50aW5nIGF0IGEgbGxhbWEuY3BwIGNoZWNr"
        "b3V0IikKCiAgICBkaXJlY3QgPSB7ImYxNiI6ICJmMTYiLCAiYmYxNiI6ICJiZjE2IiwgInE4"
        "XzAiOiAicThfMCJ9CiAgICBpZiBnZ3VmX3R5cGUgaW4gZGlyZWN0OgogICAgICAgIHRhcmdl"
        "dCA9IG91dCAvIGYiYXRyYS00Yi17Z2d1Zl90eXBlfS5nZ3VmIgogICAgICAgIHJ1bihbc3lz"
        "LmV4ZWN1dGFibGUsIHN0cihjb252ZXJ0ZXIpLCBzdHIobWVyZ2VkKSwgIi0tb3V0ZmlsZSIs"
        "IHN0cih0YXJnZXQpLCAiLS1vdXR0eXBlIiwgZGlyZWN0W2dndWZfdHlwZV1dKQogICAgICAg"
        "IHJldHVybiB0YXJnZXQKCiAgICAjIEFueXRoaW5nIGVsc2UgZ29lcyB0aHJvdWdoIGxsYW1h"
        "LXF1YW50aXplIGZyb20gYW4gZjE2IGludGVybWVkaWF0ZS4KICAgIGludGVybWVkaWF0ZSA9"
        "IG91dCAvICJhdHJhLTRiLWYxNi5nZ3VmIgogICAgcnVuKFtzeXMuZXhlY3V0YWJsZSwgc3Ry"
        "KGNvbnZlcnRlciksIHN0cihtZXJnZWQpLCAiLS1vdXRmaWxlIiwgc3RyKGludGVybWVkaWF0"
        "ZSksICItLW91dHR5cGUiLCAiZjE2Il0pCiAgICBxdWFudGl6ZSA9IGZpbmRfcXVhbnRpemUo"
        "bGxhbWFfY3BwKQogICAgaWYgcXVhbnRpemUgaXMgTm9uZToKICAgICAgICByYWlzZSBTeXN0"
        "ZW1FeGl0KAogICAgICAgICAgICAibGxhbWEtcXVhbnRpemUgYmluYXJ5IG5vdCBmb3VuZDsg"
        "YnVpbGQgbGxhbWEuY3BwIChjbWFrZSAtQiBidWlsZCAmJiBjbWFrZSAtLWJ1aWxkIGJ1aWxk"
        "ICIKICAgICAgICAgICAgIi0tdGFyZ2V0IGxsYW1hLXF1YW50aXplKSBvciB1c2UgLS1nZ3Vm"
        "IHE4XzAsIHdoaWNoIG5lZWRzIG5vIGJpbmFyeSIKICAgICAgICApCiAgICB0YXJnZXQgPSBv"
        "dXQgLyBmImF0cmEtNGIte2dndWZfdHlwZX0uZ2d1ZiIKICAgIHJ1bihbc3RyKHF1YW50aXpl"
        "KSwgc3RyKGludGVybWVkaWF0ZSksIHN0cih0YXJnZXQpLCBnZ3VmX3R5cGUudXBwZXIoKV0p"
        "CiAgICBpbnRlcm1lZGlhdGUudW5saW5rKG1pc3Npbmdfb2s9VHJ1ZSkKICAgIHJldHVybiB0"
        "YXJnZXQKCgpkZWYgZmluZF9xdWFudGl6ZShsbGFtYV9jcHA6IFBhdGgpIC0+IFBhdGggfCBO"
        "b25lOgogICAgY2FuZGlkYXRlcyA9IFsKICAgICAgICBsbGFtYV9jcHAgLyAiYnVpbGQiIC8g"
        "ImJpbiIgLyAibGxhbWEtcXVhbnRpemUiLAogICAgICAgIGxsYW1hX2NwcCAvICJidWlsZCIg"
        "LyAiYmluIiAvICJsbGFtYS1xdWFudGl6ZS5leGUiLAogICAgICAgIGxsYW1hX2NwcCAvICJs"
        "bGFtYS1xdWFudGl6ZSIsCiAgICBdCiAgICBmb3IgY2FuZGlkYXRlIGluIGNhbmRpZGF0ZXM6"
        "CiAgICAgICAgaWYgY2FuZGlkYXRlLmV4aXN0cygpOgogICAgICAgICAgICByZXR1cm4gY2Fu"
        "ZGlkYXRlCiAgICBmb3VuZCA9IHNodXRpbC53aGljaCgibGxhbWEtcXVhbnRpemUiKQogICAg"
        "cmV0dXJuIFBhdGgoZm91bmQpIGlmIGZvdW5kIGVsc2UgTm9uZQoKCmRlZiBydW4oY29tbWFu"
        "ZDogbGlzdFtzdHJdKSAtPiBOb25lOgogICAgcHJpbnQoIiQiLCAiICIuam9pbihjb21tYW5k"
        "KSkKICAgIHN1YnByb2Nlc3MucnVuKGNvbW1hbmQsIGNoZWNrPVRydWUpCgoKZGVmIG1haW4o"
        "KSAtPiBpbnQ6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlw"
        "dGlvbj0iTWVyZ2UgYW5kIGV4cG9ydCBBVFJBLTRCIikKICAgIHBhcnNlci5hZGRfYXJndW1l"
        "bnQoIi0tYWRhcHRlciIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSwgaGVscD0idHJhaW4u"
        "cHkgb3V0cHV0IGRpcmVjdG9yeSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dCIs"
        "IHR5cGU9UGF0aCwgZGVmYXVsdD1QYXRoKCJleHBvcnQiKSkKICAgIHBhcnNlci5hZGRfYXJn"
        "dW1lbnQoIi0tYmFzZSIsIGRlZmF1bHQ9b3MuZW52aXJvbi5nZXQoIkFUUkFfRlVMTF9CQVNF"
        "IiwgREVGQVVMVF9GVUxMX0JBU0UpKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1sbGFt"
        "YS1jcHAiLCB0eXBlPVBhdGgsIGRlZmF1bHQ9Tm9uZSwgaGVscD0ibGxhbWEuY3BwIGNoZWNr"
        "b3V0IGZvciBHR1VGIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZ2d1ZiIsIGRlZmF1"
        "bHQ9Tm9uZSwgaGVscD0iZjE2IHwgYmYxNiB8IHE4XzAgfCBxNF9rX20gfCBxNV9rX20gLi4u"
        "IikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc2tpcC1tZXJnZSIsIGFjdGlvbj0ic3Rv"
        "cmVfdHJ1ZSIsIGhlbHA9InJldXNlIDxvdXQ+L21lcmdlZCBmcm9tIGEgcHJldmlvdXMgcnVu"
        "IikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tY3R4IiwgdHlwZT1pbnQsIGRlZmF1bHQ9"
        "NDA5NiwgaGVscD0ibnVtX2N0eCB3cml0dGVuIGludG8gdGhlIE1vZGVsZmlsZSIpCiAgICBh"
        "cmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQoKICAgIHRyYWluaW5nID0gbG9hZF90cmFpbmlu"
        "Z19tYW5pZmVzdChhcmdzLmFkYXB0ZXIpCiAgICBhcmdzLm91dC5ta2RpcihwYXJlbnRzPVRy"
        "dWUsIGV4aXN0X29rPVRydWUpCgogICAgc3RhcnRlZCA9IHRpbWUuc3RyZnRpbWUoIiVZLSVt"
        "LSVkVCVIOiVNOiVTWiIsIHRpbWUuZ210aW1lKCkpCiAgICBtZXJnZWQgPSBhcmdzLm91dCAv"
        "ICJtZXJnZWQiCiAgICBpZiBhcmdzLnNraXBfbWVyZ2U6CiAgICAgICAgaWYgbm90IG1lcmdl"
        "ZC5leGlzdHMoKToKICAgICAgICAgICAgcmFpc2UgU3lzdGVtRXhpdChmIi0tc2tpcC1tZXJn"
        "ZSBnaXZlbiBidXQge21lcmdlZH0gZG9lcyBub3QgZXhpc3QiKQogICAgZWxzZToKICAgICAg"
        "ICBtZXJnZWQgPSBtZXJnZShhcmdzLmFkYXB0ZXIsIGFyZ3MuYmFzZSwgYXJncy5vdXQpCgog"
        "ICAgZ2d1Zl9wYXRoOiBQYXRoIHwgTm9uZSA9IE5vbmUKICAgIGlmIGFyZ3MuZ2d1ZjoKICAg"
        "ICAgICBpZiBhcmdzLmxsYW1hX2NwcCBpcyBOb25lOgogICAgICAgICAgICByYWlzZSBTeXN0"
        "ZW1FeGl0KCItLWdndWYgbmVlZHMgLS1sbGFtYS1jcHAgPHBhdGggdG8gbGxhbWEuY3BwIGNo"
        "ZWNrb3V0PiIpCiAgICAgICAgZ2d1Zl9wYXRoID0gY29udmVydF9nZ3VmKG1lcmdlZCwgYXJn"
        "cy5vdXQsIGFyZ3MubGxhbWFfY3BwLCBhcmdzLmdndWYubG93ZXIoKSkKICAgICAgICAoYXJn"
        "cy5vdXQgLyAiTW9kZWxmaWxlIikud3JpdGVfdGV4dCgKICAgICAgICAgICAgTU9ERUxGSUxF"
        "LmZvcm1hdChnZ3VmPWdndWZfcGF0aC5uYW1lLCBjdHg9YXJncy5jdHgpLCBlbmNvZGluZz0i"
        "dXRmLTgiCiAgICAgICAgKQoKICAgIG1hbmlmZXN0ID0gewogICAgICAgICJleHBvcnRlZF9h"
        "dCI6IHN0YXJ0ZWQsCiAgICAgICAgImFkYXB0ZXIiOiBzdHIoYXJncy5hZGFwdGVyKSwKICAg"
        "ICAgICAidHJhaW5pbmdfbWFuaWZlc3QiOiB0cmFpbmluZywKICAgICAgICAiZnVsbF9iYXNl"
        "IjogYXJncy5iYXNlLAogICAgICAgICJtZXJnZWRfZGlyIjogc3RyKG1lcmdlZCksCiAgICAg"
        "ICAgImdndWYiOiBOb25lCiAgICAgICAgaWYgZ2d1Zl9wYXRoIGlzIE5vbmUKICAgICAgICBl"
        "bHNlIHsiZmlsZSI6IGdndWZfcGF0aC5uYW1lLCAidHlwZSI6IGFyZ3MuZ2d1Zi5sb3dlcigp"
        "LCAic2hhMjU2Ijogc2hhMjU2X2ZpbGUoZ2d1Zl9wYXRoKSwgImJ5dGVzIjogZ2d1Zl9wYXRo"
        "LnN0YXQoKS5zdF9zaXplfSwKICAgICAgICAibm90ZSI6ICgKICAgICAgICAgICAgIldlaWdo"
        "dHMgYXJlIHRoZSBiYXNlIG1vZGVsIHBsdXMgdGhlIGFkYXB0ZXIgbmFtZWQgYWJvdmUuIFdo"
        "ZXRoZXIgdGhpcyBleHBvcnQgIgogICAgICAgICAgICAibWF5IGJlIGNhbGxlZCBBVFJBLTRC"
        "IGlzIGRlY2lkZWQgYnkgZXZhbHVhdGUucHkgYWdhaW5zdCB0aGUgdGhyZXNob2xkcyBpbiAi"
        "CiAgICAgICAgICAgICJjb25maWcvZGVmYXVsdC55YW1sLCBub3QgYnkgdGhlIGZhY3QgdGhh"
        "dCBpdCBleGlzdHMuIgogICAgICAgICksCiAgICB9CiAgICAoYXJncy5vdXQgLyAiZXhwb3J0"
        "LW1hbmlmZXN0Lmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMobWFuaWZlc3QsIGluZGVu"
        "dD0yKSwgZW5jb2Rpbmc9InV0Zi04IikKICAgIHByaW50KGYiXG5leHBvcnQgbWFuaWZlc3Q6"
        "IHthcmdzLm91dCAvICdleHBvcnQtbWFuaWZlc3QuanNvbid9IikKICAgIGlmIGdndWZfcGF0"
        "aCBpcyBub3QgTm9uZToKICAgICAgICBwcmludChmImdndWYgICAgICAgICAgIDoge2dndWZf"
        "cGF0aH0gKHttYW5pZmVzdFsnZ2d1ZiddWydieXRlcyddIC8gKDEwMjQqKjMpOi4yZn0gR2lC"
        "KSIpCiAgICByZXR1cm4gMAoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICByYWlz"
        "ZSBTeXN0ZW1FeGl0KG1haW4oKSkK"
    ),
    "config/default.yaml": (
        "IyBBVFJBLTRCIHRyYWluaW5nIGNvbmZpZ3VyYXRpb24uDQojDQojIEV2ZXJ5IGh5cGVyLXBh"
        "cmFtZXRlciBsaXZlcyBoZXJlIHNvIGEgcnVuIGlzIHJlcHJvZHVjaWJsZSBmcm9tIG9uZSBm"
        "aWxlIHBsdXMgYQ0KIyBkYXRhc2V0IGhhc2guIEFueXRoaW5nIGFic2VudCBmcm9tIHRoaXMg"
        "ZmlsZSBpcyBhIGRlZmF1bHQgaW4gdHJhaW4ucHkgYW5kIGlzDQojIHJlY29yZGVkIGluIHRo"
        "ZSBydW4gbWFuaWZlc3QgYW55d2F5Lg0KDQpydW46DQogIG5hbWU6IGF0cmEtNGItdjANCiAg"
        "c2VlZDogNDINCiAgIyBXcml0dGVuIGludG8gdGhlIG1hbmlmZXN0IHNvIGEgY2hlY2twb2lu"
        "dCBjYW4gYWx3YXlzIGJlIHRyYWNlZCBiYWNrLg0KICBub3RlczogImZpcnN0IEFUUkEtNEIg"
        "U0ZUIGF0dGVtcHQiDQoNCm1vZGVsOg0KICAjIE92ZXJyaWRhYmxlIHdpdGggQVRSQV9CQVNF"
        "LiBUaGUgNC1iaXQgdmFyaWFudCBpcyBuYW1lZCBleHBsaWNpdGx5OiBsZXR0aW5nDQogICMg"
        "dGhlIGxvYWRlciBwaWNrIGl0cyBvd24gcXVhbnRpc2F0aW9uIHNpbGVudGx5IGNoYW5nZXMg"
        "bWVtb3J5IHVzZS4NCiAgYmFzZTogdW5zbG90aC9Rd2VuMy00Qi1JbnN0cnVjdC0yNTA3LWJu"
        "Yi00Yml0DQogIG1heF9zZXFfbGVuZ3RoOiAyMDQ4DQogIGxvYWRfaW5fNGJpdDogdHJ1ZQ0K"
        "ICAjIGZwMTYgb24gYSBUNCAobm8gYmYxNik7IGJmMTYgb24gQW1wZXJlIGFuZCBuZXdlci4g"
        "ImF1dG8iIGRldGVjdHMuDQogIHByZWNpc2lvbjogYXV0bw0KDQpsb3JhOg0KICByOiAxNg0K"
        "ICBhbHBoYTogMTYNCiAgZHJvcG91dDogMC4wDQogIHRhcmdldF9tb2R1bGVzOg0KICAgIC0g"
        "cV9wcm9qDQogICAgLSBrX3Byb2oNCiAgICAtIHZfcHJvag0KICAgIC0gb19wcm9qDQogICAg"
        "LSBnYXRlX3Byb2oNCiAgICAtIHVwX3Byb2oNCiAgICAtIGRvd25fcHJvag0KICAjIFVuc2xv"
        "dGgncyBjaGVja3BvaW50aW5nIHZhcmlhbnQ7IGZhbGxzIGJhY2sgdG8gdGhlIHN0YW5kYXJk"
        "IG9uZSBlbHNld2hlcmUuDQogIGdyYWRpZW50X2NoZWNrcG9pbnRpbmc6IHVuc2xvdGgNCg0K"
        "dHJhaW5pbmc6DQogICMgOTk4IG9wdGltaXplciBzdGVwczogdHdvIHBhc3NlcyBvdmVyIHRo"
        "ZSAzLDk4NSB0cmFpbmluZyBleGFtcGxlcyB0aGF0DQogICMgYHB5dGhvbiAtbSBkYXRhLmJ1"
        "aWxkIC0tc2VlZCA0MiAtLXBlci1kb21haW4gMTAwMGAgcHJvZHVjZXMsIGF0IGFuDQogICMg"
        "ZWZmZWN0aXZlIGJhdGNoIG9mIDguIFR3byBlcG9jaHMgb3ZlciBhIGxhcmdlciBidWlsZCBy"
        "YXRoZXIgdGhhbiB0ZW4NCiAgIyBlcG9jaHMgb3ZlciB0aGUgb2xkIDgwMCDigJQgdGhlIGdl"
        "bmVyYXRvciBpcyBkZXRlcm1pbmlzdGljIGFuZCB1bmJvdW5kZWQsDQogICMgYW5kIHRlbiBw"
        "YXNzZXMgb3ZlciB0aGUgc2FtZSB0ZW1wbGF0ZXMgdGVhY2hlcyB0aGUgdGVtcGxhdGVzLg0K"
        "ICBlcG9jaHM6IDINCiAgIyBNZWFzdXJlZCBvbiBLYWdnbGUncyBUZXNsYSBUNCAoYXRyYTEy"
        "L2F0cmEtNGItdGhyb3VnaHB1dC1iZW5jaCksIGVmZmVjdGl2ZQ0KICAjIGJhdGNoIGhlbGQg"
        "YXQgOCwgYW5kIHRoZSBwZXItc3RlcCBsb3NzZXMgY2FtZSBiYWNrIGlkZW50aWNhbCBhdCBl"
        "dmVyeQ0KICAjIHNldHRpbmcsIHNvIHRoaXMgaXMgYSBwdXJlIHNwZWVkL21lbW9yeSBjaG9p"
        "Y2U6DQogICMgICBiYXRjaCAxICAyMS40MiBzL3N0ZXAgICA0Ljc5IEdpQiAgICAgIGJhdGNo"
        "IDQgIDE3LjcwIHMvc3RlcCAgIDguODcgR2lCDQogICMgICBiYXRjaCAyICAxOC42NyBzL3N0"
        "ZXAgICA2LjE4IEdpQiAgICAgIGJhdGNoIDggIDE3LjM4IHMvc3RlcCAgMTMuOTkgR2lCDQog"
        "ICMgQmF0Y2ggOCBidXlzIDIgJSBvdmVyIGJhdGNoIDQgYW5kIGxlYXZlcyAxIEdpQiBvZiBh"
        "IDE1IEdpQiBjYXJkIHdpdGggdGhlDQogICMgdmFsaWRhdGlvbiBwYXNzZXMgc3RpbGwgdG8g"
        "cnVuIGluc2lkZSBpdC4gQmF0Y2ggNCBpcyB0aGUgbGFzdCBzZXR0aW5nIHRoYXQNCiAgIyBp"
        "cyBib3RoIGZhc3QgYW5kIG5vdCBhdCB0aGUgZWRnZTogOTk4IHN0ZXBzIGluIDQgaCA1NSBt"
        "Lg0KICBwZXJfZGV2aWNlX2JhdGNoX3NpemU6IDQNCiAgZ3JhZGllbnRfYWNjdW11bGF0aW9u"
        "X3N0ZXBzOiAyDQogIHBlcl9kZXZpY2VfZXZhbF9iYXRjaF9zaXplOiA0DQogIGxlYXJuaW5n"
        "X3JhdGU6IDAuMDAwMg0KICBscl9zY2hlZHVsZXI6IGNvc2luZQ0KICB3YXJtdXBfcmF0aW86"
        "IDAuMDMNCiAgd2VpZ2h0X2RlY2F5OiAwLjAxDQogIG9wdGltaXplcjogcGFnZWRfYWRhbXdf"
        "OGJpdA0KICBtYXhfZ3JhZF9ub3JtOiAxLjANCiAgbG9nZ2luZ19zdGVwczogMTANCiAgIyBD"
        "aGVja3BvaW50IGFuZCBtZWFzdXJlIHZhbGlkYXRpb24gbG9zcyBldmVyeSBodW5kcmVkIHN0"
        "ZXBzOiB0ZW4gcG9pbnRzIGlzDQogICMgYSBjdXJ2ZSB0aGF0IGNhbiBkaXN0aW5ndWlzaCAi"
        "c3RpbGwgbGVhcm5pbmciIGZyb20gInN0YXJ0ZWQgbWVtb3Jpc2luZyIsDQogICMgd2hlcmUg"
        "dHdvIGVwb2NoIGJvdW5kYXJpZXMgYXJlIHR3byBwb2ludHMgYW5kIGNhbm5vdC4NCiAgc2F2"
        "ZV9zdGVwczogMTAwDQogIHNhdmVfdG90YWxfbGltaXQ6IDINCiAgIyBUcmFpbmluZyBvbiBj"
        "b21wbGV0aW9ucyBvbmx5OiB0aGUgbG9zcyBpZ25vcmVzIHRoZSBwcm9tcHQsIHNvIHRoZSBt"
        "b2RlbA0KICAjIGxlYXJucyB0byBwcm9kdWNlIGFuc3dlcnMgcmF0aGVyIHRoYW4gdG8gcHJl"
        "ZGljdCB0aGUgZXZpZGVuY2UgaXQgd2FzIGdpdmVuLg0KICB0cmFpbl9vbl9jb21wbGV0aW9u"
        "c19vbmx5OiB0cnVlDQoNCmRhdGE6DQogIGRpcmVjdG9yeTogZGF0YS9vdXQNCiAgdHJhaW5f"
        "ZmlsZTogdHJhaW4uanNvbmwNCiAgdmFsaWRhdGlvbl9maWxlOiB2YWxpZGF0aW9uLmpzb25s"
        "DQogIHRlc3RfZmlsZTogdGVzdC5qc29ubA0KICAjIFRoZSBidWlsZCByZWZ1c2VzIHRvIHBy"
        "b2NlZWQgYmVsb3cgdGhpcywgYmVjYXVzZSBhIG1vZGVsIHRoYXQgaGFzIHJhcmVseQ0KICAj"
        "IHNlZW4gYSByZWZ1c2FsIHdpbGwgbm90IHByb2R1Y2Ugb25lIHdoZW4gaXQgbWF0dGVycy4N"
        "CiAgbWluX25vX2FjdGlvbl9yYXRpbzogMC40MA0KDQpodWI6DQogICMgU2V0IEFUUkFfUFVT"
        "SD0xIGFuZCBIRl9UT0tFTiB0byBlbmFibGUuIE9mZiBieSBkZWZhdWx0IHNvIGEgcnVuIG5l"
        "dmVyDQogICMgdXBsb2FkcyBhbnl0aGluZyB0aGUgb3BlcmF0b3IgZGlkIG5vdCBhc2sgaXQg"
        "dG8uDQogIHB1c2g6IGZhbHNlDQogIHJlcG9faWQ6ICIiDQogIHByaXZhdGU6IHRydWUNCg0K"
        "ZXZhbHVhdGlvbjoNCiAgIyBUaHJlc2hvbGRzIGEgY2hlY2twb2ludCBtdXN0IGNsZWFyIGJl"
        "Zm9yZSBpdCBtYXkgYmUgY2FsbGVkIEFUUkEtNEIgdjAuMS4NCiAgIyBUaGV5IGFyZSBkZWxp"
        "YmVyYXRlbHkgYWJvdXQgY29ycmVjdG5lc3Mgb2YgZm9ybSBhbmQgcmVmdXNhbCwgbm90IHBy"
        "b2ZpdC4NCiAgdGhyZXNob2xkczoNCiAgICBzdHJ1Y3R1cmVkX291dHB1dF92YWxpZGl0eTog"
        "MC45OA0KICAgIHRvb2xfc2VsZWN0aW9uX2FjY3VyYWN5OiAwLjg1DQogICAgdG9vbF9hcmd1"
        "bWVudF92YWxpZGl0eTogMC45NQ0KICAgIHN0YWxlX2RhdGFfcmVqZWN0aW9uOiAwLjkwDQog"
        "ICAgaGFsbHVjaW5hdGVkX3ByaWNlX3JhdGVfbWF4OiAwLjAyDQogICAgbm9fYWN0aW9uX2Nv"
        "cnJlY3RuZXNzOiAwLjg1DQogICAgdW5zdXBwb3J0ZWRfY2hhaW5fcmVqZWN0aW9uOiAwLjk1"
        "DQogICAgbHBfYWN0aW9uX3ZhbGlkaXR5OiAwLjkwDQo="
    ),
    "data/__init__.py": (
        ""
    ),
    "data/schema.py": (
        "IiIiVHJhaW5pbmctZXhhbXBsZSBzY2hlbWEgYW5kIHZhbGlkYXRvciBmb3IgQVRSQS00Qi4N"
        "Cg0KRXZlcnkgZXhhbXBsZSBpcyBhIGNvbnZlcnNhdGlvbiB0aGF0IGVuZHMgaW4gYSBzdHJ1"
        "Y3R1cmVkIGRlY2lzaW9uLiBUd28gcnVsZXMNCnNoYXBlIHRoZSB3aG9sZSBzY2hlbWEsIGFu"
        "ZCB0aGUgdmFsaWRhdG9yIGVuZm9yY2VzIGJvdGg6DQoNCjEuICoqTm90aGluZyBpbiB0aGUg"
        "cHJvbXB0IG1heSBwb3N0ZGF0ZSB0aGUgZGVjaXNpb24uKiogQSBoaXN0b3JpY2FsIGV4YW1w"
        "bGUNCiAgIHRoYXQgbGVha3MgdG9tb3Jyb3cncyBwcmljZSB0ZWFjaGVzIHRoZSBtb2RlbCB0"
        "byBleHBlY3QgaW5mb3JtYXRpb24gaXQgd2lsbA0KICAgbmV2ZXIgaGF2ZSBhdCBpbmZlcmVu"
        "Y2UgdGltZSwgYW5kIHRoZSByZXN1bHRpbmcgZXZhbHVhdGlvbiBudW1iZXJzIGFyZQ0KICAg"
        "ZmljdGlvbi4gVGhlIG91dGNvbWUgbGl2ZXMgaW4gYSBzZXBhcmF0ZSBmaWVsZCB0aGF0IGlz"
        "IG5ldmVyIHJlbmRlcmVkLg0KDQoyLiAqKlJlZnVzYWwgaXMgYSBmaXJzdC1jbGFzcyBsYWJl"
        "bC4qKiBgYE5PX0FDVElPTmBgIGFuZCBgYElOU1VGRklDSUVOVF9EQVRBYGANCiAgIGFyZSBj"
        "b3JyZWN0IGFuc3dlcnMsIG5vdCBmYWlsdXJlcywgYW5kIHRoZSBkYXRhc2V0IGlzIHJlcXVp"
        "cmVkIHRvIGNvbnRhaW4NCiAgIGVub3VnaCBvZiB0aGVtIHRoYXQgdGhlIG1vZGVsIGxlYXJu"
        "cyB0byBwcm9kdWNlIHRoZW0uDQoiIiINCg0KZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5v"
        "dGF0aW9ucw0KDQppbXBvcnQganNvbg0KaW1wb3J0IHJlDQpmcm9tIGRhdGFjbGFzc2VzIGlt"
        "cG9ydCBkYXRhY2xhc3MsIGZpZWxkLCBhc2RpY3QNCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRh"
        "dGV0aW1lLCB0aW1lem9uZQ0KZnJvbSBlbnVtIGltcG9ydCBFbnVtDQpmcm9tIHR5cGluZyBp"
        "bXBvcnQgQW55LCBJdGVyYWJsZQ0KDQoNClNDSEVNQV9WRVJTSU9OID0gMQ0KDQpDSEFJTlMg"
        "PSAoImJhc2UiLCAiYnNjIiwgInJvYmluaG9vZCIsICJzb2xhbmEiKQ0KDQoNCmNsYXNzIERv"
        "bWFpbihzdHIsIEVudW0pOg0KICAgICIiIlRoZSBzZXZlbiBkb21haW5zIHRoZSBkYXRhc2V0"
        "IGNvdmVycy4iIiINCg0KICAgIFRPT0xfVVNFID0gInRvb2xfdXNlIg0KICAgIE1BUktFVF9S"
        "RUFTT05JTkcgPSAibWFya2V0X3JlYXNvbmluZyINCiAgICBPTkNIQUlOX1JFQVNPTklORyA9"
        "ICJvbmNoYWluX3JlYXNvbmluZyINCiAgICBUUkFESU5HX0RFQ0lTSU9OID0gInRyYWRpbmdf"
        "ZGVjaXNpb24iDQogICAgTFBfUkVBU09OSU5HID0gImxwX3JlYXNvbmluZyINCiAgICBSSVNL"
        "X1JFQVNPTklORyA9ICJyaXNrX3JlYXNvbmluZyINCiAgICBDSEFJTl9LTk9XTEVER0UgPSAi"
        "Y2hhaW5fa25vd2xlZGdlIg0KDQoNCmNsYXNzIFRyYWRlQWN0aW9uKHN0ciwgRW51bSk6DQog"
        "ICAgTk9fQUNUSU9OID0gIk5PX0FDVElPTiINCiAgICBPUEVOID0gIk9QRU4iDQogICAgUkVE"
        "VUNFID0gIlJFRFVDRSINCiAgICBDTE9TRSA9ICJDTE9TRSINCiAgICBTV0FQID0gIlNXQVAi"
        "DQoNCg0KY2xhc3MgTHBBY3Rpb24oc3RyLCBFbnVtKToNCiAgICBIT0xEID0gIkhPTEQiDQog"
        "ICAgQUREX0xJUVVJRElUWSA9ICJBRERfTElRVUlESVRZIg0KICAgIFJFTU9WRV9MSVFVSURJ"
        "VFkgPSAiUkVNT1ZFX0xJUVVJRElUWSINCiAgICBSRUJBTEFOQ0UgPSAiUkVCQUxBTkNFIg0K"
        "ICAgIENPTExFQ1RfRkVFUyA9ICJDT0xMRUNUX0ZFRVMiDQogICAgRVhJVCA9ICJFWElUIg0K"
        "DQoNCmNsYXNzIFJlc2VhcmNoU3RhdHVzKHN0ciwgRW51bSk6DQogICAgT0sgPSAiT0siDQog"
        "ICAgSU5TVUZGSUNJRU5UX0RBVEEgPSAiSU5TVUZGSUNJRU5UX0RBVEEiDQoNCg0KQGRhdGFj"
        "bGFzcw0KY2xhc3MgTWVzc2FnZToNCiAgICByb2xlOiBzdHIgICMgc3lzdGVtIHwgdXNlciB8"
        "IGFzc2lzdGFudCB8IHRvb2wNCiAgICBjb250ZW50OiBzdHINCiAgICB0b29sX2NhbGxfaWQ6"
        "IHN0ciB8IE5vbmUgPSBOb25lDQogICAgbmFtZTogc3RyIHwgTm9uZSA9IE5vbmUNCg0KDQpA"
        "ZGF0YWNsYXNzDQpjbGFzcyBUb29sU3BlYzoNCiAgICAiIiJBIHRvb2wgdGhlIG1vZGVsIG1h"
        "eSBjYWxsLCBpbiBKU09OLXNjaGVtYSBmb3JtLiIiIg0KDQogICAgbmFtZTogc3RyDQogICAg"
        "ZGVzY3JpcHRpb246IHN0cg0KICAgIHBhcmFtZXRlcnM6IGRpY3Rbc3RyLCBBbnldDQoNCg0K"
        "QGRhdGFjbGFzcw0KY2xhc3MgT3V0Y29tZToNCiAgICAiIiJXaGF0IGFjdHVhbGx5IGhhcHBl"
        "bmVkIGFmdGVyd2FyZHMuDQoNCiAgICBIZWxkIHNlcGFyYXRlbHkgZnJvbSB0aGUgY29udmVy"
        "c2F0aW9uIGFuZCBuZXZlciByZW5kZXJlZCBpbnRvIGEgcHJvbXB0LiBJdA0KICAgIGV4aXN0"
        "cyBzbyBldmFsdWF0aW9uIGNhbiBhc2sgIndhcyB0aGUgcmVmdXNhbCBjb3JyZWN0PyIgd2l0"
        "aG91dCB0aGUgbW9kZWwNCiAgICBoYXZpbmcgYmVlbiBzaG93biB0aGUgYW5zd2VyLg0KICAg"
        "ICIiIg0KDQogICAgcmVhbGl6ZWRfcmV0dXJuX2JwczogaW50IHwgTm9uZSA9IE5vbmUNCiAg"
        "ICB3YXNfY29ycmVjdDogYm9vbCB8IE5vbmUgPSBOb25lDQogICAgbm90ZXM6IHN0ciA9ICIi"
        "DQoNCg0KQGRhdGFjbGFzcw0KY2xhc3MgRXhhbXBsZToNCiAgICBpZDogc3RyDQogICAgZG9t"
        "YWluOiBEb21haW4NCiAgICBzY2hlbWFfdmVyc2lvbjogaW50ID0gU0NIRU1BX1ZFUlNJT04N"
        "Cg0KICAgIGNoYWluOiBzdHIgfCBOb25lID0gTm9uZQ0KICAgICMgVGhlIGluc3RhbnQgdGhl"
        "IGRlY2lzaW9uIGlzIG1hZGUuIE5vdGhpbmcgaW4gYG1lc3NhZ2VzYCBtYXkgYmUgbmV3ZXIu"
        "DQogICAgZGVjaXNpb25fdGltZTogc3RyID0gIiINCiAgICBjcmVhdGVkX2F0OiBzdHIgPSAi"
        "Ig0KDQogICAgbWVzc2FnZXM6IGxpc3RbTWVzc2FnZV0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rv"
        "cnk9bGlzdCkNCiAgICB0b29sczogbGlzdFtUb29sU3BlY10gPSBmaWVsZChkZWZhdWx0X2Zh"
        "Y3Rvcnk9bGlzdCkNCg0KICAgICMgVGhlIHN0cnVjdHVyZWQgYW5zd2VyIHRoZSBtb2RlbCBz"
        "aG91bGQgcHJvZHVjZSwgYXMgYSBkaWN0Lg0KICAgIGV4cGVjdGVkX291dHB1dDogZGljdFtz"
        "dHIsIEFueV0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCkNCg0KICAgIG91dGNvbWU6"
        "IE91dGNvbWUgfCBOb25lID0gTm9uZQ0KDQogICAgcHJvdmVuYW5jZTogc3RyID0gInN5bnRo"
        "ZXRpYyINCiAgICBsaWNlbnNlOiBzdHIgPSAiTUlUIg0KICAgIHF1YWxpdHlfc2NvcmU6IGZs"
        "b2F0ID0gMS4wDQogICAgc3BsaXQ6IHN0ciA9ICJ0cmFpbiIgICMgdHJhaW4gfCB2YWxpZGF0"
        "aW9uIHwgdGVzdA0KDQogICAgIyBFeHBsaWNpdCBiZWhhdmlvdXIgbGFiZWxzLCBlLmcuICJz"
        "dGFsZSIsICJ1bnN1cHBvcnRlZC1jaGFpbiIsICJyZWZ1c2FsIi4NCiAgICAjIEV2YWx1YXRp"
        "b24gc2VsZWN0cyBvbiB0aGVzZSByYXRoZXIgdGhhbiBncmVwcGluZyB0aGUgdGV4dCwgd2hp"
        "Y2ggaXMgaG93IGENCiAgICAjIG1ldHJpYyBlbmRzIHVwIHNjb3JpbmcgInRocmVzaG9sZCIg"
        "YXMgYSBzdGFsZW5lc3MgY2FzZS4NCiAgICB0YWdzOiBsaXN0W3N0cl0gPSBmaWVsZChkZWZh"
        "dWx0X2ZhY3Rvcnk9bGlzdCkNCg0KICAgIGRlZiB0b19qc29uKHNlbGYpIC0+IHN0cjoNCiAg"
        "ICAgICAgcmV0dXJuIGpzb24uZHVtcHMoYXNkaWN0KHNlbGYpLCBkZWZhdWx0PXN0ciwgc29y"
        "dF9rZXlzPVRydWUpDQoNCg0KY2xhc3MgVmFsaWRhdGlvbkVycm9yKEV4Y2VwdGlvbik6DQog"
        "ICAgIiIiUmFpc2VkIGZvciBhbiBleGFtcGxlIHRoYXQgbXVzdCBub3QgZW50ZXIgdGhlIGRh"
        "dGFzZXQuIiIiDQoNCg0KSVNPID0gIiVZLSVtLSVkVCVIOiVNOiVTWiINCg0KIyBBbnkgdGlt"
        "ZXN0YW1wLWxvb2tpbmcgc3RyaW5nIGluc2lkZSBhIHJlbmRlcmVkIG1lc3NhZ2UuDQpUSU1F"
        "U1RBTVBfUEFUVEVSTiA9IHJlLmNvbXBpbGUociJcZHs0fS1cZHsyfS1cZHsyfVRcZHsyfTpc"
        "ZHsyfTpcZHsyfVoiKQ0KDQojIFBocmFzZXMgdGhhdCBnaXZlIGF3YXkgYSBmdXR1cmUgb3V0"
        "Y29tZS4gQSBwcm9tcHQgY29udGFpbmluZyBvbmUgb2YgdGhlc2UgaXMNCiMgdGVhY2hpbmcg"
        "dGhlIG1vZGVsIHRvIHJlYWQgdGhlIGFuc3dlciBvZmYgdGhlIHF1ZXN0aW9uLg0KTEVBS0FH"
        "RV9QSFJBU0VTID0gKA0KICAgICJpbiBoaW5kc2lnaHQiLA0KICAgICJpdCB0dXJuZWQgb3V0"
        "IiwNCiAgICAidGhlIHByaWNlIGxhdGVyIiwNCiAgICAid291bGQgaGF2ZSByZXR1cm5lZCIs"
        "DQogICAgImFzIHdlIG5vdyBrbm93IiwNCiAgICAidGhlIGNvcnJlY3QgYW5zd2VyIGlzIiwN"
        "CiAgICAiZXZlbnR1YWxseSByb3NlIiwNCiAgICAiZXZlbnR1YWxseSBmZWxsIiwNCikNCg0K"
        "DQpkZWYgcGFyc2VfdGltZSh2YWx1ZTogc3RyKSAtPiBkYXRldGltZToNCiAgICByZXR1cm4g"
        "ZGF0ZXRpbWUuc3RycHRpbWUodmFsdWUsIElTTykucmVwbGFjZSh0emluZm89dGltZXpvbmUu"
        "dXRjKQ0KDQoNCmRlZiB2YWxpZGF0ZShleGFtcGxlOiBFeGFtcGxlKSAtPiBOb25lOg0KICAg"
        "ICIiIlJhaXNlIDpjbGFzczpgVmFsaWRhdGlvbkVycm9yYCBpZiB0aGUgZXhhbXBsZSBpcyB1"
        "bnVzYWJsZS4NCg0KICAgIERlbGliZXJhdGVseSBzdHJpY3QuIEEgZGF0YXNldCBpcyB0aGUg"
        "b25lIGFydGVmYWN0IHdob3NlIGRlZmVjdHMgYXJlDQogICAgaW52aXNpYmxlIGluIHRoZSBm"
        "aW5hbCBtb2RlbCwgc28gYW55dGhpbmcgYW1iaWd1b3VzIGlzIHJlamVjdGVkIHJhdGhlciB0"
        "aGFuDQogICAgcmVwYWlyZWQuDQogICAgIiIiDQogICAgaWYgZXhhbXBsZS5zY2hlbWFfdmVy"
        "c2lvbiAhPSBTQ0hFTUFfVkVSU0lPTjoNCiAgICAgICAgcmFpc2UgVmFsaWRhdGlvbkVycm9y"
        "KGYie2V4YW1wbGUuaWR9OiB1bnN1cHBvcnRlZCBzY2hlbWEgdmVyc2lvbiIpDQoNCiAgICBp"
        "ZiBub3QgZXhhbXBsZS5pZDoNCiAgICAgICAgcmFpc2UgVmFsaWRhdGlvbkVycm9yKCJhbiBl"
        "eGFtcGxlIGhhcyBubyBpZCIpDQoNCiAgICBpZiBleGFtcGxlLmNoYWluIGlzIG5vdCBOb25l"
        "IGFuZCBleGFtcGxlLmNoYWluIG5vdCBpbiBDSEFJTlM6DQogICAgICAgIHJhaXNlIFZhbGlk"
        "YXRpb25FcnJvcihmIntleGFtcGxlLmlkfTogdW5zdXBwb3J0ZWQgY2hhaW4ge2V4YW1wbGUu"
        "Y2hhaW4hcn0iKQ0KDQogICAgaWYgbm90IGV4YW1wbGUuZGVjaXNpb25fdGltZToNCiAgICAg"
        "ICAgcmFpc2UgVmFsaWRhdGlvbkVycm9yKGYie2V4YW1wbGUuaWR9OiBkZWNpc2lvbl90aW1l"
        "IGlzIHJlcXVpcmVkIikNCg0KICAgIHRyeToNCiAgICAgICAgZGVjaXNpb25fYXQgPSBwYXJz"
        "ZV90aW1lKGV4YW1wbGUuZGVjaXNpb25fdGltZSkNCiAgICBleGNlcHQgVmFsdWVFcnJvciBh"
        "cyBlcnJvcjoNCiAgICAgICAgcmFpc2UgVmFsaWRhdGlvbkVycm9yKGYie2V4YW1wbGUuaWR9"
        "OiBtYWxmb3JtZWQgZGVjaXNpb25fdGltZSIpIGZyb20gZXJyb3INCg0KICAgIGlmIG5vdCBl"
        "eGFtcGxlLm1lc3NhZ2VzOg0KICAgICAgICByYWlzZSBWYWxpZGF0aW9uRXJyb3IoZiJ7ZXhh"
        "bXBsZS5pZH06IG5vIG1lc3NhZ2VzIikNCg0KICAgIHJvbGVzID0gW21lc3NhZ2Uucm9sZSBm"
        "b3IgbWVzc2FnZSBpbiBleGFtcGxlLm1lc3NhZ2VzXQ0KICAgIGlmIHJvbGVzWzBdICE9ICJz"
        "eXN0ZW0iOg0KICAgICAgICByYWlzZSBWYWxpZGF0aW9uRXJyb3IoZiJ7ZXhhbXBsZS5pZH06"
        "IHRoZSBmaXJzdCBtZXNzYWdlIG11c3QgYmUgdGhlIHN5c3RlbSBwcm9tcHQiKQ0KICAgIGlm"
        "IHJvbGVzWy0xXSAhPSAiYXNzaXN0YW50IjoNCiAgICAgICAgcmFpc2UgVmFsaWRhdGlvbkVy"
        "cm9yKGYie2V4YW1wbGUuaWR9OiB0aGUgbGFzdCBtZXNzYWdlIG11c3QgYmUgdGhlIGFzc2lz"
        "dGFudCBhbnN3ZXIiKQ0KICAgIGZvciByb2xlIGluIHJvbGVzOg0KICAgICAgICBpZiByb2xl"
        "IG5vdCBpbiAoInN5c3RlbSIsICJ1c2VyIiwgImFzc2lzdGFudCIsICJ0b29sIik6DQogICAg"
        "ICAgICAgICByYWlzZSBWYWxpZGF0aW9uRXJyb3IoZiJ7ZXhhbXBsZS5pZH06IHVua25vd24g"
        "cm9sZSB7cm9sZSFyfSIpDQoNCiAgICBfY2hlY2tfbm9fZnV0dXJlX2xlYWthZ2UoZXhhbXBs"
        "ZSwgZGVjaXNpb25fYXQpDQogICAgX2NoZWNrX2V4cGVjdGVkX291dHB1dChleGFtcGxlKQ0K"
        "DQogICAgaWYgZXhhbXBsZS5zcGxpdCBub3QgaW4gKCJ0cmFpbiIsICJ2YWxpZGF0aW9uIiwg"
        "InRlc3QiKToNCiAgICAgICAgcmFpc2UgVmFsaWRhdGlvbkVycm9yKGYie2V4YW1wbGUuaWR9"
        "OiB1bmtub3duIHNwbGl0IHtleGFtcGxlLnNwbGl0IXJ9IikNCg0KDQpkZWYgX2NoZWNrX25v"
        "X2Z1dHVyZV9sZWFrYWdlKGV4YW1wbGU6IEV4YW1wbGUsIGRlY2lzaW9uX2F0OiBkYXRldGlt"
        "ZSkgLT4gTm9uZToNCiAgICAiIiJSZWZ1c2UgYW55IHByb21wdCB0aGF0IGNvbnRhaW5zIGlu"
        "Zm9ybWF0aW9uIGZyb20gYWZ0ZXIgdGhlIGRlY2lzaW9uLiIiIg0KICAgIHByb21wdF9tZXNz"
        "YWdlcyA9IFttIGZvciBtIGluIGV4YW1wbGUubWVzc2FnZXMgaWYgbS5yb2xlICE9ICJhc3Np"
        "c3RhbnQiXQ0KDQogICAgZm9yIG1lc3NhZ2UgaW4gcHJvbXB0X21lc3NhZ2VzOg0KICAgICAg"
        "ICBsb3dlcmVkID0gbWVzc2FnZS5jb250ZW50Lmxvd2VyKCkNCiAgICAgICAgZm9yIHBocmFz"
        "ZSBpbiBMRUFLQUdFX1BIUkFTRVM6DQogICAgICAgICAgICBpZiBwaHJhc2UgaW4gbG93ZXJl"
        "ZDoNCiAgICAgICAgICAgICAgICByYWlzZSBWYWxpZGF0aW9uRXJyb3IoDQogICAgICAgICAg"
        "ICAgICAgICAgIGYie2V4YW1wbGUuaWR9OiBwcm9tcHQgY29udGFpbnMgb3V0Y29tZSBsYW5n"
        "dWFnZSAoe3BocmFzZSFyfSkiDQogICAgICAgICAgICAgICAgKQ0KDQogICAgICAgIGZvciBt"
        "YXRjaCBpbiBUSU1FU1RBTVBfUEFUVEVSTi5maW5kYWxsKG1lc3NhZ2UuY29udGVudCk6DQog"
        "ICAgICAgICAgICB0cnk6DQogICAgICAgICAgICAgICAgc3RhbXAgPSBwYXJzZV90aW1lKG1h"
        "dGNoKQ0KICAgICAgICAgICAgZXhjZXB0IFZhbHVlRXJyb3I6DQogICAgICAgICAgICAgICAg"
        "Y29udGludWUNCiAgICAgICAgICAgIGlmIHN0YW1wID4gZGVjaXNpb25fYXQ6DQogICAgICAg"
        "ICAgICAgICAgcmFpc2UgVmFsaWRhdGlvbkVycm9yKA0KICAgICAgICAgICAgICAgICAgICBm"
        "IntleGFtcGxlLmlkfTogcHJvbXB0IHJlZmVyZW5jZXMge21hdGNofSwgYWZ0ZXIgZGVjaXNp"
        "b25fdGltZSINCiAgICAgICAgICAgICAgICApDQoNCiAgICAjIFRoZSBvdXRjb21lIG11c3Qg"
        "bmV2ZXIgYXBwZWFyIGluIHRoZSBjb252ZXJzYXRpb24gYXQgYWxsLg0KICAgIGlmIGV4YW1w"
        "bGUub3V0Y29tZSBpcyBub3QgTm9uZSBhbmQgZXhhbXBsZS5vdXRjb21lLm5vdGVzOg0KICAg"
        "ICAgICByZW5kZXJlZCA9ICIgIi5qb2luKG0uY29udGVudCBmb3IgbSBpbiBleGFtcGxlLm1l"
        "c3NhZ2VzKS5sb3dlcigpDQogICAgICAgIGlmIGV4YW1wbGUub3V0Y29tZS5ub3Rlcy5sb3dl"
        "cigpWzo0MF0gaW4gcmVuZGVyZWQ6DQogICAgICAgICAgICByYWlzZSBWYWxpZGF0aW9uRXJy"
        "b3IoZiJ7ZXhhbXBsZS5pZH06IHRoZSBvdXRjb21lIG5vdGUgYXBwZWFycyBpbiB0aGUgY29u"
        "dmVyc2F0aW9uIikNCg0KDQpkZWYgX2NoZWNrX2V4cGVjdGVkX291dHB1dChleGFtcGxlOiBF"
        "eGFtcGxlKSAtPiBOb25lOg0KICAgICIiIlRoZSBhbnN3ZXIgbXVzdCBtYXRjaCB0aGUgZG9t"
        "YWluJ3Mgc2NoZW1hIGV4YWN0bHkuIiIiDQogICAgb3V0cHV0ID0gZXhhbXBsZS5leHBlY3Rl"
        "ZF9vdXRwdXQNCiAgICBpZiBub3QgaXNpbnN0YW5jZShvdXRwdXQsIGRpY3QpIG9yIG5vdCBv"
        "dXRwdXQ6DQogICAgICAgIHJhaXNlIFZhbGlkYXRpb25FcnJvcihmIntleGFtcGxlLmlkfTog"
        "ZXhwZWN0ZWRfb3V0cHV0IG11c3QgYmUgYSBub24tZW1wdHkgb2JqZWN0IikNCg0KICAgIGlm"
        "IGV4YW1wbGUuZG9tYWluIGlzIERvbWFpbi5UUkFESU5HX0RFQ0lTSU9OOg0KICAgICAgICBh"
        "Y3Rpb24gPSBvdXRwdXQuZ2V0KCJhY3Rpb24iKQ0KICAgICAgICBpZiBhY3Rpb24gbm90IGlu"
        "IHthLnZhbHVlIGZvciBhIGluIFRyYWRlQWN0aW9ufToNCiAgICAgICAgICAgIHJhaXNlIFZh"
        "bGlkYXRpb25FcnJvcihmIntleGFtcGxlLmlkfTogaW52YWxpZCB0cmFkZSBhY3Rpb24ge2Fj"
        "dGlvbiFyfSIpDQogICAgICAgIGZvciBrZXkgaW4gKCJjaGFpbiIsICJyZWFzb24iLCAiY29u"
        "ZmlkZW5jZSIpOg0KICAgICAgICAgICAgaWYga2V5IG5vdCBpbiBvdXRwdXQ6DQogICAgICAg"
        "ICAgICAgICAgcmFpc2UgVmFsaWRhdGlvbkVycm9yKGYie2V4YW1wbGUuaWR9OiB0cmFkaW5n"
        "IG91dHB1dCBpcyBtaXNzaW5nIHtrZXl9IikNCiAgICAgICAgY29uZmlkZW5jZSA9IG91dHB1"
        "dC5nZXQoImNvbmZpZGVuY2UiKQ0KICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShjb25maWRl"
        "bmNlLCAoaW50LCBmbG9hdCkpIG9yIG5vdCAwLjAgPD0gZmxvYXQoY29uZmlkZW5jZSkgPD0g"
        "MS4wOg0KICAgICAgICAgICAgcmFpc2UgVmFsaWRhdGlvbkVycm9yKGYie2V4YW1wbGUuaWR9"
        "OiBjb25maWRlbmNlIG11c3QgYmUgYmV0d2VlbiAwIGFuZCAxIikNCiAgICAgICAgIyBBIHJl"
        "ZnVzYWwgdGhhdCBjbGFpbXMgaGlnaCBjb25maWRlbmNlIGlzIGluY29oZXJlbnQgYW5kIHRl"
        "YWNoZXMgdGhlDQogICAgICAgICMgbW9kZWwgdGhhdCB0aGUgZmllbGQgbWVhbnMgbm90aGlu"
        "Zy4NCiAgICAgICAgaWYgYWN0aW9uID09IFRyYWRlQWN0aW9uLk5PX0FDVElPTi52YWx1ZSBh"
        "bmQgZmxvYXQoY29uZmlkZW5jZSkgPiAwLjk6DQogICAgICAgICAgICByYWlzZSBWYWxpZGF0"
        "aW9uRXJyb3IoZiJ7ZXhhbXBsZS5pZH06IE5PX0FDVElPTiB3aXRoIG5lYXItY2VydGFpbiBj"
        "b25maWRlbmNlIikNCg0KICAgIGVsaWYgZXhhbXBsZS5kb21haW4gaXMgRG9tYWluLkxQX1JF"
        "QVNPTklORzoNCiAgICAgICAgYWN0aW9uID0gb3V0cHV0LmdldCgiYWN0aW9uIikNCiAgICAg"
        "ICAgaWYgYWN0aW9uIG5vdCBpbiB7YS52YWx1ZSBmb3IgYSBpbiBMcEFjdGlvbn06DQogICAg"
        "ICAgICAgICByYWlzZSBWYWxpZGF0aW9uRXJyb3IoZiJ7ZXhhbXBsZS5pZH06IGludmFsaWQg"
        "TFAgYWN0aW9uIHthY3Rpb24hcn0iKQ0KDQogICAgZWxpZiBleGFtcGxlLmRvbWFpbiBpcyBE"
        "b21haW4uVE9PTF9VU0U6DQogICAgICAgIGlmICJ0b29sIiBub3QgaW4gb3V0cHV0IGFuZCAi"
        "c3RhdHVzIiBub3QgaW4gb3V0cHV0Og0KICAgICAgICAgICAgcmFpc2UgVmFsaWRhdGlvbkVy"
        "cm9yKGYie2V4YW1wbGUuaWR9OiB0b29sLXVzZSBvdXRwdXQgbmVlZHMgYSB0b29sIG9yIGEg"
        "c3RhdHVzIikNCiAgICAgICAgdG9vbCA9IG91dHB1dC5nZXQoInRvb2wiKQ0KICAgICAgICBp"
        "ZiB0b29sIGlzIG5vdCBOb25lOg0KICAgICAgICAgICAgZGVjbGFyZWQgPSB7c3BlYy5uYW1l"
        "IGZvciBzcGVjIGluIGV4YW1wbGUudG9vbHN9DQogICAgICAgICAgICBpZiB0b29sIG5vdCBp"
        "biBkZWNsYXJlZDoNCiAgICAgICAgICAgICAgICByYWlzZSBWYWxpZGF0aW9uRXJyb3IoZiJ7"
        "ZXhhbXBsZS5pZH06IGNhbGxzIHVuZGVjbGFyZWQgdG9vbCB7dG9vbCFyfSIpDQoNCiAgICBl"
        "bGlmIGV4YW1wbGUuZG9tYWluIGluIChEb21haW4uTUFSS0VUX1JFQVNPTklORywgRG9tYWlu"
        "Lk9OQ0hBSU5fUkVBU09OSU5HKToNCiAgICAgICAgc3RhdHVzID0gb3V0cHV0LmdldCgic3Rh"
        "dHVzIikNCiAgICAgICAgaWYgc3RhdHVzIG5vdCBpbiB7cy52YWx1ZSBmb3IgcyBpbiBSZXNl"
        "YXJjaFN0YXR1c306DQogICAgICAgICAgICByYWlzZSBWYWxpZGF0aW9uRXJyb3IoZiJ7ZXhh"
        "bXBsZS5pZH06IGludmFsaWQgcmVzZWFyY2ggc3RhdHVzIHtzdGF0dXMhcn0iKQ0KDQoNCmRl"
        "ZiB2YWxpZGF0ZV9hbGwoZXhhbXBsZXM6IEl0ZXJhYmxlW0V4YW1wbGVdKSAtPiBsaXN0W3N0"
        "cl06DQogICAgIiIiVmFsaWRhdGUgYSBjb2xsZWN0aW9uLCByZXR1cm5pbmcgZXZlcnkgZXJy"
        "b3IgcmF0aGVyIHRoYW4gdGhlIGZpcnN0LiIiIg0KICAgIGVycm9yczogbGlzdFtzdHJdID0g"
        "W10NCiAgICBzZWVuX2lkczogc2V0W3N0cl0gPSBzZXQoKQ0KDQogICAgZm9yIGV4YW1wbGUg"
        "aW4gZXhhbXBsZXM6DQogICAgICAgIGlmIGV4YW1wbGUuaWQgaW4gc2Vlbl9pZHM6DQogICAg"
        "ICAgICAgICBlcnJvcnMuYXBwZW5kKGYie2V4YW1wbGUuaWR9OiBkdXBsaWNhdGUgaWQiKQ0K"
        "ICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgc2Vlbl9pZHMuYWRkKGV4YW1wbGUuaWQp"
        "DQoNCiAgICAgICAgdHJ5Og0KICAgICAgICAgICAgdmFsaWRhdGUoZXhhbXBsZSkNCiAgICAg"
        "ICAgZXhjZXB0IFZhbGlkYXRpb25FcnJvciBhcyBlcnJvcjoNCiAgICAgICAgICAgIGVycm9y"
        "cy5hcHBlbmQoc3RyKGVycm9yKSkNCg0KICAgIHJldHVybiBlcnJvcnMNCg=="
    ),
    "data/build.py": (
        "..."
        "U3lzdGVtRXhpdChtYWluKCkpDQo="
    ),
    "data/checks.py": (
        "IiIiRGF0YXNldCBxdWFsaXR5IGdhdGVzLg0KDQpUaGVzZSBydW4gYmVmb3JlIGFueSB0cmFp"
        "bmluZyBhbmQgZmFpbCB0aGUgYnVpbGQgcmF0aGVyIHRoYW4gd2Fybi4gQSBkYXRhc2V0DQpk"
        "ZWZlY3QgaXMgaW52aXNpYmxlIGluIHRoZSB0cmFpbmVkIG1vZGVsIGJ1dCBzaG93cyB1cCBh"
        "cyBhIGNvbmZpZGVudCB3cm9uZw0KYW5zd2VyIG1vbnRocyBsYXRlciwgc28gdGhlIGNoZWNr"
        "cyBhcmUgZGVsaWJlcmF0ZWx5IHVuZm9yZ2l2aW5nLg0KDQpXaGF0IGlzIGNoZWNrZWQ6DQoN"
        "Ci0gKipMZWFrYWdlKiog4oCUIG5vdGhpbmcgaW4gYSBwcm9tcHQgbWF5IHBvc3RkYXRlIGl0"
        "cyBkZWNpc2lvbiB0aW1lLCBhbmQgbm8NCiAgaGlzdG9yaWNhbCBleGFtcGxlIG1heSBhcHBl"
        "YXIgaW4gYSBzcGxpdCBlYXJsaWVyIHRoYW4gb25lIGl0IGNocm9ub2xvZ2ljYWxseQ0KICBm"
        "b2xsb3dzLg0KLSAqKkR1cGxpY2F0aW9uKiog4oCUIGV4YWN0IGFuZCBuZWFyLWR1cGxpY2F0"
        "ZSBwcm9tcHRzIGluZmxhdGUgYXBwYXJlbnQgZGF0YXNldA0KICBzaXplIGFuZCBsZXQgdGhl"
        "IG1vZGVsIG1lbW9yaXNlIGluc3RlYWQgb2YgZ2VuZXJhbGlzZS4NCi0gKipCYWxhbmNlKiog"
        "4oCUIHJlZnVzYWxzIG11c3QgYmUgd2VsbCByZXByZXNlbnRlZC4gQSBtb2RlbCB0aGF0IGhh"
        "cyByYXJlbHkgc2Vlbg0KICBgYE5PX0FDVElPTmBgIHdpbGwgbm90IHByb2R1Y2UgaXQgd2hl"
        "biBpdCBtYXR0ZXJzLg0KLSAqKkNoYWluIGNvdmVyYWdlKiog4oCUIGFsbCBmb3VyIGNoYWlu"
        "cyBtdXN0IGFwcGVhciwgc28gdGhlIG1vZGVsIGRvZXMgbm90IGxlYXJuDQogIHRoYXQgImNo"
        "YWluIiBtZWFucyBCYXNlLg0KIiIiDQoNCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3Rh"
        "dGlvbnMNCg0KaW1wb3J0IGFyZ3BhcnNlDQppbXBvcnQgaGFzaGxpYg0KaW1wb3J0IGpzb24N"
        "CmltcG9ydCBzeXMNCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IENvdW50ZXINCmZyb20gZGF0"
        "YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcw0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQpm"
        "cm9tIHR5cGluZyBpbXBvcnQgSXRlcmFibGUNCg0KZnJvbSAuc2NoZW1hIGltcG9ydCAoDQog"
        "ICAgQ0hBSU5TLA0KICAgIERvbWFpbiwNCiAgICBFeGFtcGxlLA0KICAgIE1lc3NhZ2UsDQog"
        "ICAgT3V0Y29tZSwNCiAgICBUb29sU3BlYywNCiAgICBUcmFkZUFjdGlvbiwNCiAgICBwYXJz"
        "ZV90aW1lLA0KICAgIHZhbGlkYXRlX2FsbCwNCikNCg0KDQojIEEgZGF0YXNldCB3aXRoIGZl"
        "d2VyIHJlZnVzYWxzIHRoYW4gdGhpcyB0ZWFjaGVzIHRoZSBtb2RlbCB0aGF0IGRvaW5nIG5v"
        "dGhpbmcNCiMgaXMgYW4gZWRnZSBjYXNlLiBJdCBpcyB0aGUgc2luZ2xlIG1vc3QgaW1wb3J0"
        "YW50IGJlaGF2aW91ciBBVFJBIG5lZWRzLg0KTUlOX05PX0FDVElPTl9SQVRJTyA9IDAuNDAN"
        "Cg0KIyBOZWFyLWR1cGxpY2F0ZSB0aHJlc2hvbGQgb24gdG9rZW4tc2V0IHNpbWlsYXJpdHku"
        "DQpNQVhfSkFDQ0FSRCA9IDAuOTANCg0KDQpAZGF0YWNsYXNzDQpjbGFzcyBDaGVja1JlcG9y"
        "dDoNCiAgICB0b3RhbDogaW50DQogICAgZXJyb3JzOiBsaXN0W3N0cl0NCiAgICB3YXJuaW5n"
        "czogbGlzdFtzdHJdDQogICAgc3RhdHM6IGRpY3Rbc3RyLCBvYmplY3RdDQoNCiAgICBAcHJv"
        "cGVydHkNCiAgICBkZWYgb2soc2VsZikgLT4gYm9vbDoNCiAgICAgICAgcmV0dXJuIG5vdCBz"
        "ZWxmLmVycm9ycw0KDQogICAgZGVmIHJlbmRlcihzZWxmKSAtPiBzdHI6DQogICAgICAgIGxp"
        "bmVzID0gW2YiZXhhbXBsZXM6IHtzZWxmLnRvdGFsfSJdDQogICAgICAgIGZvciBrZXksIHZh"
        "bHVlIGluIHNvcnRlZChzZWxmLnN0YXRzLml0ZW1zKCkpOg0KICAgICAgICAgICAgbGluZXMu"
        "YXBwZW5kKGYiICB7a2V5fToge3ZhbHVlfSIpDQogICAgICAgIGlmIHNlbGYud2FybmluZ3M6"
        "DQogICAgICAgICAgICBsaW5lcy5hcHBlbmQoZiJ3YXJuaW5ncyAoe2xlbihzZWxmLndhcm5p"
        "bmdzKX0pOiIpDQogICAgICAgICAgICBsaW5lcy5leHRlbmQoZiIgIC0ge3dhcm5pbmd9IiBm"
        "b3Igd2FybmluZyBpbiBzZWxmLndhcm5pbmdzWzoyMF0pDQogICAgICAgIGlmIHNlbGYuZXJy"
        "b3JzOg0KICAgICAgICAgICAgbGluZXMuYXBwZW5kKGYiRVJST1JTICh7bGVuKHNlbGYuZXJy"
        "b3JzKX0pOiIpDQogICAgICAgICAgICBsaW5lcy5leHRlbmQoZiIgIC0ge2Vycm9yfSIgZm9y"
        "IGVycm9yIGluIHNlbGYuZXJyb3JzWzo0MF0pDQogICAgICAgIGVsc2U6DQogICAgICAgICAg"
        "ICBsaW5lcy5hcHBlbmQoIm5vIGVycm9ycyIpDQogICAgICAgIHJldHVybiAiXG4iLmpvaW4o"
        "bGluZXMpDQoNCg0KZGVmIGxvYWRfanNvbmwocGF0aDogUGF0aCkgLT4gbGlzdFtFeGFtcGxl"
        "XToNCiAgICBleGFtcGxlczogbGlzdFtFeGFtcGxlXSA9IFtdDQoNCiAgICBmb3IgbGluZV9u"
        "dW1iZXIsIGxpbmUgaW4gZW51bWVyYXRlKHBhdGgucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYt"
        "OCIpLnNwbGl0bGluZXMoKSwgc3RhcnQ9MSk6DQogICAgICAgIGxpbmUgPSBsaW5lLnN0cmlw"
        "KCkNCiAgICAgICAgaWYgbm90IGxpbmU6DQogICAgICAgICAgICBjb250aW51ZQ0KICAgICAg"
        "ICB0cnk6DQogICAgICAgICAgICByYXcgPSBqc29uLmxvYWRzKGxpbmUpDQogICAgICAgIGV4"
        "Y2VwdCBqc29uLkpTT05EZWNvZGVFcnJvciBhcyBlcnJvcjoNCiAgICAgICAgICAgIHJhaXNl"
        "IFZhbHVlRXJyb3IoZiJ7cGF0aH06e2xpbmVfbnVtYmVyfTogbWFsZm9ybWVkIEpTT04iKSBm"
        "cm9tIGVycm9yDQogICAgICAgIGV4YW1wbGVzLmFwcGVuZChfZnJvbV9kaWN0KHJhdykpDQoN"
        "CiAgICByZXR1cm4gZXhhbXBsZXMNCg0KDQpkZWYgX2Zyb21fZGljdChyYXc6IGRpY3QpIC0+"
        "IEV4YW1wbGU6DQogICAgb3V0Y29tZSA9IHJhdy5nZXQoIm91dGNvbWUiKQ0KICAgIHJldHVy"
        "biBFeGFtcGxlKA0KICAgICAgICBpZD1yYXdbImlkIl0sDQogICAgICAgIGRvbWFpbj1Eb21h"
        "aW4ocmF3WyJkb21haW4iXSksDQogICAgICAgIHNjaGVtYV92ZXJzaW9uPXJhdy5nZXQoInNj"
        "aGVtYV92ZXJzaW9uIiwgMSksDQogICAgICAgIGNoYWluPXJhdy5nZXQoImNoYWluIiksDQog"
        "ICAgICAgIGRlY2lzaW9uX3RpbWU9cmF3LmdldCgiZGVjaXNpb25fdGltZSIsICIiKSwNCiAg"
        "ICAgICAgY3JlYXRlZF9hdD1yYXcuZ2V0KCJjcmVhdGVkX2F0IiwgIiIpLA0KICAgICAgICBt"
        "ZXNzYWdlcz1bTWVzc2FnZSgqKm1lc3NhZ2UpIGZvciBtZXNzYWdlIGluIHJhdy5nZXQoIm1l"
        "c3NhZ2VzIiwgW10pXSwNCiAgICAgICAgdG9vbHM9W1Rvb2xTcGVjKCoqdG9vbCkgZm9yIHRv"
        "b2wgaW4gcmF3LmdldCgidG9vbHMiLCBbXSldLA0KICAgICAgICBleHBlY3RlZF9vdXRwdXQ9"
        "cmF3LmdldCgiZXhwZWN0ZWRfb3V0cHV0Iiwge30pLA0KICAgICAgICBvdXRjb21lPU91dGNv"
        "bWUoKipvdXRjb21lKSBpZiBvdXRjb21lIGVsc2UgTm9uZSwNCiAgICAgICAgcHJvdmVuYW5j"
        "ZT1yYXcuZ2V0KCJwcm92ZW5hbmNlIiwgInN5bnRoZXRpYyIpLA0KICAgICAgICBsaWNlbnNl"
        "PXJhdy5nZXQoImxpY2Vuc2UiLCAiTUlUIiksDQogICAgICAgIHF1YWxpdHlfc2NvcmU9cmF3"
        "LmdldCgicXVhbGl0eV9zY29yZSIsIDEuMCksDQogICAgICAgIHNwbGl0PXJhdy5nZXQoInNw"
        "bGl0IiwgInRyYWluIiksDQogICAgICAgIHRhZ3M9bGlzdChyYXcuZ2V0KCJ0YWdzIiwgW10p"
        "KSwNCiAgICApDQoNCg0KZGVmIGNoZWNrKGV4YW1wbGVzOiBsaXN0W0V4YW1wbGVdKSAtPiBD"
        "aGVja1JlcG9ydDoNCiAgICBlcnJvcnMgPSB2YWxpZGF0ZV9hbGwoZXhhbXBsZXMpDQogICAg"
        "d2FybmluZ3M6IGxpc3Rbc3RyXSA9IFtdDQoNCiAgICBlcnJvcnMuZXh0ZW5kKF9jaGVja19k"
        "dXBsaWNhdGVzKGV4YW1wbGVzKSkNCiAgICBlcnJvcnMuZXh0ZW5kKF9jaGVja19jaHJvbm9s"
        "b2dpY2FsX3NwbGl0cyhleGFtcGxlcykpDQogICAgd2FybmluZ3MuZXh0ZW5kKF9jaGVja19u"
        "ZWFyX2R1cGxpY2F0ZXMoZXhhbXBsZXMpKQ0KDQogICAgc3RhdHMgPSBfc3RhdGlzdGljcyhl"
        "eGFtcGxlcykNCg0KICAgIHJlZnVzYWxfcmF0aW8gPSBzdGF0cy5nZXQoIm5vX2FjdGlvbl9y"
        "YXRpbyIsIDAuMCkNCiAgICBpZiBpc2luc3RhbmNlKHJlZnVzYWxfcmF0aW8sIGZsb2F0KSBh"
        "bmQgc3RhdHMuZ2V0KCJ0cmFkaW5nX2V4YW1wbGVzIiwgMCk6DQogICAgICAgIGlmIHJlZnVz"
        "YWxfcmF0aW8gPCBNSU5fTk9fQUNUSU9OX1JBVElPOg0KICAgICAgICAgICAgZXJyb3JzLmFw"
        "cGVuZCgNCiAgICAgICAgICAgICAgICBmIm9ubHkge3JlZnVzYWxfcmF0aW86LjAlfSBvZiB0"
        "cmFkaW5nIGV4YW1wbGVzIGFyZSBOT19BQ1RJT047ICINCiAgICAgICAgICAgICAgICBmImF0"
        "IGxlYXN0IHtNSU5fTk9fQUNUSU9OX1JBVElPOi4wJX0gaXMgcmVxdWlyZWQiDQogICAgICAg"
        "ICAgICApDQoNCiAgICBtaXNzaW5nX2NoYWlucyA9IFtjaGFpbiBmb3IgY2hhaW4gaW4gQ0hB"
        "SU5TIGlmIHN0YXRzLmdldChmImNoYWluX3tjaGFpbn0iLCAwKSA9PSAwXQ0KICAgIGlmIG1p"
        "c3NpbmdfY2hhaW5zOg0KICAgICAgICBlcnJvcnMuYXBwZW5kKGYibm8gZXhhbXBsZXMgZm9y"
        "IGNoYWluKHMpOiB7JywgJy5qb2luKG1pc3NpbmdfY2hhaW5zKX0iKQ0KDQogICAgZm9yIHNw"
        "bGl0IGluICgidHJhaW4iLCAidmFsaWRhdGlvbiIsICJ0ZXN0Iik6DQogICAgICAgIGlmIHN0"
        "YXRzLmdldChmInNwbGl0X3tzcGxpdH0iLCAwKSA9PSAwOg0KICAgICAgICAgICAgZXJyb3Jz"
        "LmFwcGVuZChmInNwbGl0IHtzcGxpdCFyfSBpcyBlbXB0eSIpDQoNCiAgICByZXR1cm4gQ2hl"
        "Y2tSZXBvcnQodG90YWw9bGVuKGV4YW1wbGVzKSwgZXJyb3JzPWVycm9ycywgd2FybmluZ3M9"
        "d2FybmluZ3MsIHN0YXRzPXN0YXRzKQ0KDQoNCmRlZiBfY2hlY2tfZHVwbGljYXRlcyhleGFt"
        "cGxlczogSXRlcmFibGVbRXhhbXBsZV0pIC0+IGxpc3Rbc3RyXToNCiAgICAiIiJFeGFjdCBk"
        "dXBsaWNhdGUgcHJvbXB0cywgYnkgaGFzaCBvZiB0aGUgbm9uLWFzc2lzdGFudCB0dXJucy4i"
        "IiINCiAgICBzZWVuOiBkaWN0W3N0ciwgc3RyXSA9IHt9DQogICAgZXJyb3JzOiBsaXN0W3N0"
        "cl0gPSBbXQ0KDQogICAgZm9yIGV4YW1wbGUgaW4gZXhhbXBsZXM6DQogICAgICAgIGRpZ2Vz"
        "dCA9IF9wcm9tcHRfaGFzaChleGFtcGxlKQ0KICAgICAgICBpZiBkaWdlc3QgaW4gc2VlbjoN"
        "CiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJ7ZXhhbXBsZS5pZH06IGR1cGxpY2F0ZSBw"
        "cm9tcHQgb2Yge3NlZW5bZGlnZXN0XX0iKQ0KICAgICAgICBlbHNlOg0KICAgICAgICAgICAg"
        "c2VlbltkaWdlc3RdID0gZXhhbXBsZS5pZA0KDQogICAgcmV0dXJuIGVycm9ycw0KDQoNCmRl"
        "ZiBfY2hlY2tfbmVhcl9kdXBsaWNhdGVzKGV4YW1wbGVzOiBsaXN0W0V4YW1wbGVdKSAtPiBs"
        "aXN0W3N0cl06DQogICAgIiIiRmxhZyBwcm9tcHRzIHRoYXQgYXJlIG5lYXJseSBpZGVudGlj"
        "YWwuDQoNCiAgICBSZXBvcnRlZCBhcyB3YXJuaW5ncywgbm90IGVycm9yczogdGVtcGxhdGVk"
        "IGdlbmVyYXRpb24gbGVnaXRpbWF0ZWx5DQogICAgcHJvZHVjZXMgc2ltaWxhciBwcm9tcHRz"
        "LCBhbmQgdGhlIHJpZ2h0IHJlc3BvbnNlIGlzIHVzdWFsbHkgdG8gd2lkZW4gdGhlDQogICAg"
        "Z2VuZXJhdG9yIHJhdGhlciB0aGFuIHRvIGRyb3AgZXhhbXBsZXMuDQogICAgIiIiDQogICAg"
        "d2FybmluZ3M6IGxpc3Rbc3RyXSA9IFtdDQogICAgdG9rZW5fc2V0cyA9IFsoZXhhbXBsZS5p"
        "ZCwgX3Rva2VucyhleGFtcGxlKSkgZm9yIGV4YW1wbGUgaW4gZXhhbXBsZXNdDQoNCiAgICBm"
        "b3IgaW5kZXgsIChsZWZ0X2lkLCBsZWZ0KSBpbiBlbnVtZXJhdGUodG9rZW5fc2V0cyk6DQog"
        "ICAgICAgIGZvciByaWdodF9pZCwgcmlnaHQgaW4gdG9rZW5fc2V0c1tpbmRleCArIDEgOiBp"
        "bmRleCArIDQwXToNCiAgICAgICAgICAgIGlmIG5vdCBsZWZ0IG9yIG5vdCByaWdodDoNCiAg"
        "ICAgICAgICAgICAgICBjb250aW51ZQ0KICAgICAgICAgICAgc2ltaWxhcml0eSA9IGxlbihs"
        "ZWZ0ICYgcmlnaHQpIC8gbGVuKGxlZnQgfCByaWdodCkNCiAgICAgICAgICAgIGlmIHNpbWls"
        "YXJpdHkgPj0gTUFYX0pBQ0NBUkQ6DQogICAgICAgICAgICAgICAgd2FybmluZ3MuYXBwZW5k"
        "KA0KICAgICAgICAgICAgICAgICAgICBmIntsZWZ0X2lkfSBhbmQge3JpZ2h0X2lkfSBhcmUg"
        "e3NpbWlsYXJpdHk6LjAlfSBzaW1pbGFyIg0KICAgICAgICAgICAgICAgICkNCg0KICAgIHJl"
        "dHVybiB3YXJuaW5ncw0KDQoNCmRlZiBfY2hlY2tfY2hyb25vbG9naWNhbF9zcGxpdHMoZXhh"
        "bXBsZXM6IGxpc3RbRXhhbXBsZV0pIC0+IGxpc3Rbc3RyXToNCiAgICAiIiJIaXN0b3JpY2Fs"
        "IHNwbGl0cyBtdXN0IG5vdCBvdmVybGFwIGluIHRpbWUuDQoNCiAgICBJZiBhIHRlc3QgZXhh"
        "bXBsZSBwcmVkYXRlcyBhIHRyYWluaW5nIGV4YW1wbGUsIHRoZSBtb2RlbCBoYXMgYmVlbiB0"
        "cmFpbmVkDQogICAgb24gdGhlIGZ1dHVyZSBvZiBpdHMgb3duIHRlc3Qgc2V0IGFuZCBldmVy"
        "eSBtZXRyaWMgaXMgb3B0aW1pc3RpYy4NCiAgICAiIiINCiAgICBlcnJvcnM6IGxpc3Rbc3Ry"
        "XSA9IFtdDQogICAgYm91bmRzOiBkaWN0W3N0ciwgdHVwbGVbc3RyLCBzdHJdXSA9IHt9DQoN"
        "CiAgICBmb3Igc3BsaXQgaW4gKCJ0cmFpbiIsICJ2YWxpZGF0aW9uIiwgInRlc3QiKToNCiAg"
        "ICAgICAgc3RhbXBzID0gc29ydGVkKA0KICAgICAgICAgICAgZXhhbXBsZS5kZWNpc2lvbl90"
        "aW1lDQogICAgICAgICAgICBmb3IgZXhhbXBsZSBpbiBleGFtcGxlcw0KICAgICAgICAgICAg"
        "aWYgZXhhbXBsZS5zcGxpdCA9PSBzcGxpdCBhbmQgZXhhbXBsZS5wcm92ZW5hbmNlID09ICJo"
        "aXN0b3JpY2FsIg0KICAgICAgICApDQogICAgICAgIGlmIHN0YW1wczoNCiAgICAgICAgICAg"
        "IGJvdW5kc1tzcGxpdF0gPSAoc3RhbXBzWzBdLCBzdGFtcHNbLTFdKQ0KDQogICAgb3JkZXIg"
        "PSBbc3BsaXQgZm9yIHNwbGl0IGluICgidHJhaW4iLCAidmFsaWRhdGlvbiIsICJ0ZXN0Iikg"
        "aWYgc3BsaXQgaW4gYm91bmRzXQ0KICAgIGZvciBlYXJsaWVyLCBsYXRlciBpbiB6aXAob3Jk"
        "ZXIsIG9yZGVyWzE6XSk6DQogICAgICAgIGlmIHBhcnNlX3RpbWUoYm91bmRzW2xhdGVyXVsw"
        "XSkgPCBwYXJzZV90aW1lKGJvdW5kc1tlYXJsaWVyXVsxXSk6DQogICAgICAgICAgICBlcnJv"
        "cnMuYXBwZW5kKA0KICAgICAgICAgICAgICAgIGYiaGlzdG9yaWNhbCB7bGF0ZXJ9IHNwbGl0"
        "IHN0YXJ0cyBhdCB7Ym91bmRzW2xhdGVyXVswXX0sICINCiAgICAgICAgICAgICAgICBmImJl"
        "Zm9yZSB7ZWFybGllcn0gZW5kcyBhdCB7Ym91bmRzW2VhcmxpZXJdWzFdfSINCiAgICAgICAg"
        "ICAgICkNCg0KICAgIHJldHVybiBlcnJvcnMNCg0KDQpkZWYgX3N0YXRpc3RpY3MoZXhhbXBs"
        "ZXM6IGxpc3RbRXhhbXBsZV0pIC0+IGRpY3Rbc3RyLCBvYmplY3RdOg0KICAgIGRvbWFpbnMg"
        "PSBDb3VudGVyKGV4YW1wbGUuZG9tYWluLnZhbHVlIGZvciBleGFtcGxlIGluIGV4YW1wbGVz"
        "KQ0KICAgIHNwbGl0cyA9IENvdW50ZXIoZXhhbXBsZS5zcGxpdCBmb3IgZXhhbXBsZSBpbiBl"
        "eGFtcGxlcykNCiAgICBjaGFpbnMgPSBDb3VudGVyKGV4YW1wbGUuY2hhaW4gZm9yIGV4YW1w"
        "bGUgaW4gZXhhbXBsZXMgaWYgZXhhbXBsZS5jaGFpbikNCg0KICAgIHRyYWRpbmcgPSBbZSBm"
        "b3IgZSBpbiBleGFtcGxlcyBpZiBlLmRvbWFpbiBpcyBEb21haW4uVFJBRElOR19ERUNJU0lP"
        "Tl0NCiAgICBub19hY3Rpb24gPSBbDQogICAgICAgIGUgZm9yIGUgaW4gdHJhZGluZyBpZiBl"
        "LmV4cGVjdGVkX291dHB1dC5nZXQoImFjdGlvbiIpID09IFRyYWRlQWN0aW9uLk5PX0FDVElP"
        "Ti52YWx1ZQ0KICAgIF0NCg0KICAgIHN0YXRzOiBkaWN0W3N0ciwgb2JqZWN0XSA9IHsNCiAg"
        "ICAgICAgInRyYWRpbmdfZXhhbXBsZXMiOiBsZW4odHJhZGluZyksDQogICAgICAgICJub19h"
        "Y3Rpb25fZXhhbXBsZXMiOiBsZW4obm9fYWN0aW9uKSwNCiAgICAgICAgIm5vX2FjdGlvbl9y"
        "YXRpbyI6IChsZW4obm9fYWN0aW9uKSAvIGxlbih0cmFkaW5nKSkgaWYgdHJhZGluZyBlbHNl"
        "IDAuMCwNCiAgICB9DQoNCiAgICBmb3IgZG9tYWluLCBjb3VudCBpbiBkb21haW5zLml0ZW1z"
        "KCk6DQogICAgICAgIHN0YXRzW2YiZG9tYWluX3tkb21haW59Il0gPSBjb3VudA0KICAgIGZv"
        "ciBzcGxpdCwgY291bnQgaW4gc3BsaXRzLml0ZW1zKCk6DQogICAgICAgIHN0YXRzW2Yic3Bs"
        "aXRfe3NwbGl0fSJdID0gY291bnQNCiAgICBmb3IgY2hhaW4gaW4gQ0hBSU5TOg0KICAgICAg"
        "ICBzdGF0c1tmImNoYWluX3tjaGFpbn0iXSA9IGNoYWlucy5nZXQoY2hhaW4sIDApDQoNCiAg"
        "ICByZXR1cm4gc3RhdHMNCg0KDQpkZWYgX3Byb21wdF9oYXNoKGV4YW1wbGU6IEV4YW1wbGUp"
        "IC0+IHN0cjoNCiAgICBwcm9tcHQgPSAiXG4iLmpvaW4oDQogICAgICAgIGYie21lc3NhZ2Uu"
        "cm9sZX06e21lc3NhZ2UuY29udGVudH0iDQogICAgICAgIGZvciBtZXNzYWdlIGluIGV4YW1w"
        "bGUubWVzc2FnZXMNCiAgICAgICAgaWYgbWVzc2FnZS5yb2xlICE9ICJhc3Npc3RhbnQiDQog"
        "ICAgKQ0KICAgIHJldHVybiBoYXNobGliLnNoYTI1Nihwcm9tcHQuZW5jb2RlKCJ1dGYtOCIp"
        "KS5oZXhkaWdlc3QoKQ0KDQoNCmRlZiBfdG9rZW5zKGV4YW1wbGU6IEV4YW1wbGUpIC0+IHNl"
        "dFtzdHJdOg0KICAgIHRleHQgPSAiICIuam9pbihtLmNvbnRlbnQgZm9yIG0gaW4gZXhhbXBs"
        "ZS5tZXNzYWdlcyBpZiBtLnJvbGUgPT0gInVzZXIiKQ0KICAgIHJldHVybiBzZXQodGV4dC5s"
        "b3dlcigpLnNwbGl0KCkpDQoNCg0KZGVmIGRhdGFzZXRfaGFzaChleGFtcGxlczogbGlzdFtF"
        "eGFtcGxlXSkgLT4gc3RyOg0KICAgICIiIkEgc3RhYmxlIGhhc2ggb2YgdGhlIHdob2xlIGRh"
        "dGFzZXQsIHJlY29yZGVkIGluIGV2ZXJ5IHRyYWluaW5nIHJ1bi4iIiINCiAgICBkaWdlc3Qg"
        "PSBoYXNobGliLnNoYTI1NigpDQogICAgZm9yIGV4YW1wbGUgaW4gc29ydGVkKGV4YW1wbGVz"
        "LCBrZXk9bGFtYmRhIGl0ZW06IGl0ZW0uaWQpOg0KICAgICAgICBkaWdlc3QudXBkYXRlKGV4"
        "YW1wbGUudG9fanNvbigpLmVuY29kZSgidXRmLTgiKSkNCiAgICByZXR1cm4gZGlnZXN0Lmhl"
        "eGRpZ2VzdCgpDQoNCg0KZGVmIG1haW4oKSAtPiBpbnQ6DQogICAgcGFyc2VyID0gYXJncGFy"
        "c2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249IlZhbGlkYXRlIGFuIEFUUkEtNEIgZGF0"
        "YXNldCIpDQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiZGlyZWN0b3J5IiwgdHlwZT1QYXRo"
        "LCBoZWxwPSJkaXJlY3RvcnkgY29udGFpbmluZyAqLmpzb25sIikNCiAgICBhcmdzID0gcGFy"
        "c2VyLnBhcnNlX2FyZ3MoKQ0KDQogICAgZmlsZXMgPSBzb3J0ZWQoYXJncy5kaXJlY3Rvcnku"
        "Z2xvYigiKi5qc29ubCIpKQ0KICAgIGlmIG5vdCBmaWxlczoNCiAgICAgICAgcHJpbnQoZiJu"
        "byAuanNvbmwgZmlsZXMgZm91bmQgaW4ge2FyZ3MuZGlyZWN0b3J5fSIsIGZpbGU9c3lzLnN0"
        "ZGVycikNCiAgICAgICAgcmV0dXJuIDINCg0KICAgIGV4YW1wbGVzOiBsaXN0W0V4YW1wbGVd"
        "ID0gW10NCiAgICBmb3IgcGF0aCBpbiBmaWxlczoNCiAgICAgICAgZXhhbXBsZXMuZXh0ZW5k"
        "KGxvYWRfanNvbmwocGF0aCkpDQoNCiAgICByZXBvcnQgPSBjaGVjayhleGFtcGxlcykNCiAg"
        "ICBwcmludChyZXBvcnQucmVuZGVyKCkpDQogICAgcHJpbnQoZiJkYXRhc2V0IGhhc2g6IHtk"
        "YXRhc2V0X2hhc2goZXhhbXBsZXMpfSIpDQoNCiAgICByZXR1cm4gMCBpZiByZXBvcnQub2sg"
        "ZWxzZSAxDQoNCg0KaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoNCiAgICByYWlzZSBTeXN0"
        "ZW1FeGl0KG1haW4oKSkNCg=="
    ),
}

EXPECTED = {
    "evaluate.py": "0d0c81b0d1d119266457c0c140af29b7800087c414b3230720139a9a697383df",
    "prompting.py": "73235d389fe220ac4f3cfbd81f5f8dee488d33acbab407025161694ebcdb952b",
    "export.py": "3cd7b354626363a5a1949537f2be84e69e9feb9200aa744f2ac20f97397d1349",
    "config/default.yaml": "b11423b97a1312a449e87e619a69a12d1c9a8dac71085891b6f7c7012bce1610",
    "data/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
    "data/schema.py": "16b41bf435d2a0ec6097eba02ed60dbab0f4d7f41cf94a8f6f1c3035589b58f6",
    "data/build.py": "45f4babc91a589e9bf12d361dfcf470a11936de5e6f27300f8598f626402e2e6",
    "data/checks.py": "04b7f4f12c7c6273787d652d1a5de3729241d6639efec030e8bd165b6c9db873",
}

root = pathlib.Path('/kaggle/working/atra')
if root.exists():
    shutil.rmtree(root)

for name, blob in FILES.items():
    raw = base64.b64decode(blob)
    target = root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(raw)
    digest = hashlib.sha256(raw).hexdigest()
    match = 'MATCHES working tree' if digest == EXPECTED[name] else 'DIFFERS <-- STOP'
    print(f"{digest}  {name}  ({len(raw)} bytes)  {match}")

bad = [
    name
    for name, blob in FILES.items()
    if hashlib.sha256(base64.b64decode(blob)).hexdigest() != EXPECTED[name]
]
assert not bad, f'carried files do not match the working tree: {bad}'

print()
print(sorted(str(p.relative_to(root)) for p in root.rglob('*') if p.is_file()))
print()
print('--- evaluation thresholds this run is judged against ---')
print((root / 'config/default.yaml').read_text().split('evaluation:')[-1])

## 3. The new adapter

400 steps, final loss 0.0980891, `status: completed` — and, unlike its
parent, a `chat_template.jinja` beside the weights.

In [ ]:
import hashlib, json, pathlib, shutil

# A data source's layout inside /kaggle/input is not the folder that was
# uploaded, so the adapter is found by its files rather than by an assumed
# path. The continuation run also saved checkpoint-300/ and checkpoint-400/,
# and each of those carries its own adapter_config.json — so the directory is
# picked by the pair (adapter_config.json AND manifest.json), which only the
# final adapter has.
inputs = pathlib.Path('/kaggle/input')
print('inputs:', [str(p) for p in inputs.glob('*')])
candidates = [
    p.parent
    for p in inputs.rglob('adapter_config.json')
    if (p.parent / 'manifest.json').exists()
]
assert candidates, f'no adapter directory with a manifest under {inputs}'
assert len(candidates) == 1, f'ambiguous: {candidates}'
src = candidates[0]
print('adapter found at:', src)
print('adapter files  :', sorted(p.name for p in src.iterdir()))

dst = pathlib.Path('/kaggle/working/runs/atra-4b')
if dst.exists():
    shutil.rmtree(dst)
dst.parent.mkdir(parents=True, exist_ok=True)
dst.mkdir()
# Only the files the merge and the tokenizer need; the intermediate
# checkpoints are 126 MiB each and are not part of this artifact.
for item in sorted(src.iterdir()):
    if item.is_file():
        shutil.copy2(item, dst / item.name)

print()
print('staged        :', sorted(p.name for p in dst.iterdir()))
for name in ('adapter_model.safetensors', 'chat_template.jinja', 'adapter_config.json'):
    p = dst / name
    if p.exists():
        print(f'{hashlib.sha256(p.read_bytes()).hexdigest()}  {name}  ({p.stat().st_size} bytes)')
    else:
        print(f'{"MISSING":<64}  {name}')

TEMPLATE_IN_ADAPTER = (dst / 'chat_template.jinja').exists()
print()
print('adapter ships its own chat_template.jinja:', TEMPLATE_IN_ADAPTER,
      '  <- the old adapter did not, which is how the GGUF lost its template')

manifest = json.loads((dst / 'manifest.json').read_text())
print()
print('run name    :', manifest['run_name'])
print('status      :', manifest['status'])
print('steps       :', manifest['steps'])
print('parent steps:', manifest.get('parent_steps'))
print('final loss  :', round(manifest['final_loss'], 7))
print('base model  :', manifest['base_model'])
print('base rev    :', manifest.get('base_revision'))
print('gpu         :', manifest.get('gpu_name'))
print('dataset hash:', manifest['dataset_hash'])
print('counts      :', manifest['dataset_counts'])
assert manifest['status'] == 'completed'
assert manifest['steps'] == 400, manifest['steps']

### 3.1 Which template path will `export.py` take?

`export.py` uses the adapter's own tokenizer template when it has one and
falls back to the base model only when it does not. Which of the two
happened is worth knowing exactly, so it is established before the merge and
confirmed from the merge log afterwards.

In [ ]:
import hashlib, os, pathlib

# Which path does export.py take?
#
# export.py loads `AutoTokenizer.from_pretrained(<adapter>)` and, only if that
# tokenizer has no chat template, falls back to recovering one from the base
# model — refusing outright if neither has one. The previous adapter had no
# `chat_template.jinja`, which is how the exported GGUF ended up with no
# template and llama-server fell back to a built-in default with no `tools`
# block. train.py now writes the template beside the weights.
#
# So: does the tokenizer built from THIS adapter already carry a template? If
# yes, the fallback is not taken and the template in the GGUF is the one the
# training run rendered its prompts with, byte for byte.
os.environ.setdefault('HF_HOME', '/kaggle/temp/hf')
pathlib.Path('/kaggle/temp/hf').mkdir(parents=True, exist_ok=True)

from transformers import AutoTokenizer

ADAPTER = pathlib.Path('/kaggle/working/runs/atra-4b')
adapter_tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER))
adapter_template = getattr(adapter_tokenizer, 'chat_template', None)

print('AutoTokenizer.from_pretrained(adapter).chat_template is set:', bool(adapter_template))
assert adapter_template, (
    'the adapter tokenizer has no chat template, so export.py would take the '
    'base-model fallback; report that, it is the finding'
)
ADAPTER_TEMPLATE_SHA = hashlib.sha256(adapter_template.encode('utf-8')).hexdigest()
print('template chars :', len(adapter_template))
print('template sha256:', ADAPTER_TEMPLATE_SHA)
print('file sha256    :',
      hashlib.sha256((ADAPTER / 'chat_template.jinja').read_bytes()).hexdigest(),
      ' (chat_template.jinja on disk)')
print()
print('EXPORT PATH  : adapter carries its own template -> export.py should NOT print')
print('               "template   : recovered from ..." in the merge cell below.')

# Is it the same template the base model publishes? Same answer either way is
# fine; a difference would be worth knowing before the artifact is served.
BASE = manifest['base_model']
REVISION = manifest.get('base_revision')
base_tokenizer = AutoTokenizer.from_pretrained(BASE, revision=REVISION)
base_template = getattr(base_tokenizer, 'chat_template', None)
print()
print('base model     :', BASE, '@', REVISION)
print('base template  :', 'none' if not base_template else
      hashlib.sha256(base_template.encode('utf-8')).hexdigest())
print('identical      :', base_template == adapter_template)

# Does the adapter's template honour `tools`? Rendered both ways, in the same
# shape prompting.render_example builds, so the answer is arithmetic.
probe_messages = [
    {'role': 'system', 'content': 'system prompt'},
    {'role': 'user', 'content': 'user turn'},
]
probe_tools = [
    {
        'type': 'function',
        'function': {
            'name': 'get_market_snapshot',
            'description': 'Current price, liquidity and volume for a pool.',
            'parameters': {'type': 'object', 'required': ['chain'], 'properties': {'chain': {'type': 'string'}}},
        },
    }
]
plain = adapter_tokenizer.apply_chat_template(probe_messages, tokenize=False, add_generation_prompt=True)
with_tools = adapter_tokenizer.apply_chat_template(
    probe_messages, tools=probe_tools, tokenize=False, add_generation_prompt=True
)
print()
print('rendered without tools:', len(plain), 'chars')
print('rendered with tools   :', len(with_tools), 'chars')
print('tool name in rendering:', 'get_market_snapshot' in with_tools)
assert 'get_market_snapshot' not in plain
assert 'get_market_snapshot' in with_tools, 'this template ignores `tools`'
print()
print(with_tools[: len(with_tools) - len(plain) + 200])

## 4. llama.cpp: converter, quantizer and server

The CUDA build failed to configure on this image last time — FindCUDAToolkit
could not resolve `CUDA::cuda_driver` — so the driver library is pointed at
explicitly, with a CPU fallback that costs time and not correctness.

In [ ]:
%%bash
set -e
set -o pipefail
# /kaggle/temp does not exist unless a notebook creates it.
mkdir -p /kaggle/temp
cd /kaggle/temp
rm -rf llama.cpp
git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
cd llama.cpp
echo "llama.cpp commit: $(git rev-parse HEAD)"
# Only the converter's own package. llama.cpp's requirements file pins a CPU
# build of torch, which would replace Kaggle's CUDA torch.
pip install -q --no-deps gguf

BUILD=/kaggle/temp/llama.cpp/build
set +e
ARCH=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader 2>/dev/null | head -1 | tr -d '. ')
set -e
[ -z "${ARCH}" ] && ARCH=75
echo "cuda architecture: ${ARCH}"

# The CUDA build failed to configure on this image on the previous attempt:
# FindCUDAToolkit could not resolve CUDA::cuda_driver, because the driver
# library a container sees is libcuda.so.1 with no libcuda.so symlink and the
# toolkit's stub directory is not on the default link path. Both are pointed
# at explicitly here. It matters: the CPU server generates at ~5 tokens/s, so
# 95 prompts of a few hundred tokens each is hours rather than minutes.
echo "--- libcuda on this image ---"
ls -la /usr/local/cuda/lib64/stubs/libcuda.so 2>/dev/null || echo "no toolkit stub"
ls -la /usr/lib/x86_64-linux-gnu/libcuda.so* 2>/dev/null || echo "no driver lib"

DRIVER=""
for candidate in /usr/local/cuda/lib64/stubs/libcuda.so \
                 /usr/lib/x86_64-linux-gnu/libcuda.so \
                 /usr/lib/x86_64-linux-gnu/libcuda.so.1; do
  if [ -e "$candidate" ]; then DRIVER="$candidate"; break; fi
done
echo "driver library: ${DRIVER:-none found}"

CFG=1
BLD=1
if [ -n "$DRIVER" ]; then
  set +e
  cmake -S /kaggle/temp/llama.cpp -B "$BUILD" -DCMAKE_BUILD_TYPE=Release \
    -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES="${ARCH}" -DLLAMA_CURL=OFF \
    -DCMAKE_LIBRARY_PATH=/usr/local/cuda/lib64/stubs \
    -DCUDA_cuda_driver_LIBRARY="$DRIVER" \
    -DCUDA_CUDA_LIBRARY="$DRIVER" \
    > /kaggle/temp/cmake-configure.log 2>&1
  CFG=$?
  if [ $CFG -eq 0 ]; then
    cmake --build "$BUILD" --target llama-quantize llama-server -j"$(nproc)" \
      > /kaggle/temp/cmake-build.log 2>&1
    BLD=$?
  fi
  set -e
fi

if [ $CFG -ne 0 ] || [ $BLD -ne 0 ]; then
  echo "=== CUDA build failed (configure=$CFG build=$BLD); falling back to CPU ==="
  tail -20 /kaggle/temp/cmake-configure.log 2>/dev/null || true
  tail -20 /kaggle/temp/cmake-build.log 2>/dev/null || true
  rm -rf "$BUILD"
  cmake -S /kaggle/temp/llama.cpp -B "$BUILD" -DCMAKE_BUILD_TYPE=Release \
    -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /kaggle/temp/cmake-configure-cpu.log 2>&1
  cmake --build "$BUILD" --target llama-quantize llama-server -j"$(nproc)" \
    > /kaggle/temp/cmake-build-cpu.log 2>&1
  echo CPU > /kaggle/temp/build-kind.txt
else
  echo CUDA > /kaggle/temp/build-kind.txt
fi

echo "build kind: $(cat /kaggle/temp/build-kind.txt)"
ls -la "$BUILD/bin" | grep -E "llama-quantize|llama-server"

echo
echo "--- llama-server chat template flags ---"
"$BUILD/bin/llama-server" --help 2>&1 | grep -E -- "--jinja|--chat-template-file" \
  || echo "WARNING: this build has neither --jinja nor --chat-template-file"

## 5. Merge and convert

In [ ]:
%%bash
set -e
set -o pipefail
# Keep the 8 GiB base download, the merged fp16 weights and the f16 GGUF
# intermediate off the 20 GiB /kaggle/working quota; only the q4_k_m file and
# its manifest are worth saving as output.
export HF_HOME=/kaggle/temp/hf
mkdir -p /kaggle/temp/hf
cd /kaggle/working/atra
python export.py --adapter /kaggle/working/runs/atra-4b --out /kaggle/temp/atra-export \
  --llama-cpp /kaggle/temp/llama.cpp --gguf q4_k_m > /kaggle/working/export.log 2>&1
tail -25 /kaggle/working/export.log

echo
echo "=== which template path did export.py take? ==="
if grep -q "template   : recovered from" /kaggle/working/export.log; then
  echo "FALLBACK: export.py recovered the template from the base model"
  grep "template   : recovered from" /kaggle/working/export.log
else
  echo "ADAPTER: no 'recovered from' line -> the adapter's own chat_template.jinja was used"
fi
grep -E "^(base model|adapter|dtype|merged|template)" /kaggle/working/export.log || true

echo
ls -la /kaggle/temp/atra-export
cp /kaggle/temp/atra-export/atra-4b-q4_k_m.gguf /kaggle/working/
cp /kaggle/temp/atra-export/Modelfile /kaggle/working/
cp /kaggle/temp/atra-export/export-manifest.json /kaggle/working/
rm -rf /kaggle/temp/atra-export/merged
echo
echo "=== the artifact ==="
sha256sum /kaggle/working/atra-4b-q4_k_m.gguf
stat -c '%s bytes' /kaggle/working/atra-4b-q4_k_m.gguf
cat /kaggle/working/export-manifest.json
df -h /kaggle/working

## 6. The template inside the artifact

Everything above is about the adapter and the exporter. This opens the
q4_k_m file itself, reads `tokenizer.chat_template` out of its metadata, and
renders it both ways. If the tools do not appear here, they will not appear
for anyone who downloads this file, and the run stops.

In [ ]:
import hashlib, json, pathlib

# The template inside the artifact, read out of the artifact.
#
# Everything so far is about the adapter and the exporter. This cell opens the
# q4_k_m file itself and asks what chat template it carries, because that is
# the only copy llama-server will have once the GGUF is all anyone has. The
# previous export carried none, and nothing downstream said a word.
GGUF = pathlib.Path('/kaggle/working/atra-4b-q4_k_m.gguf')
GGUF_SHA = hashlib.sha256()
with GGUF.open('rb') as handle:
    for chunk in iter(lambda: handle.read(1 << 22), b''):
        GGUF_SHA.update(chunk)
GGUF_SHA = GGUF_SHA.hexdigest()
GGUF_BYTES = GGUF.stat().st_size
print('gguf        :', GGUF)
print('gguf sha256 :', GGUF_SHA)
print('gguf bytes  :', GGUF_BYTES, f'({GGUF_BYTES / (1024 ** 3):.3f} GiB)')

from gguf import GGUFReader

reader = GGUFReader(str(GGUF))


def field_str(field):
    try:
        value = field.contents()
        if isinstance(value, str):
            return value
        if isinstance(value, bytes):
            return value.decode('utf-8')
    except Exception:
        pass
    return bytes(field.parts[field.data[-1]]).decode('utf-8')


keys = sorted(reader.fields.keys())
print()
print('gguf metadata keys carrying a template:',
      [k for k in keys if 'template' in k.lower()])

field = reader.fields.get('tokenizer.chat_template')
assert field is not None, (
    'the GGUF carries NO tokenizer.chat_template key. That is the finding: '
    'the exported artifact still cannot be served the way it was trained.'
)
GGUF_TEMPLATE = field_str(field)
GGUF_TEMPLATE_SHA = hashlib.sha256(GGUF_TEMPLATE.encode('utf-8')).hexdigest()
print()
print('GGUF chat template chars :', len(GGUF_TEMPLATE))
print('GGUF chat template sha256:', GGUF_TEMPLATE_SHA)
print('adapter template sha256  :', ADAPTER_TEMPLATE_SHA)
print('identical to the adapter :', GGUF_TEMPLATE_SHA == ADAPTER_TEMPLATE_SHA)

# Does the template *in the file* produce a tools block? Rendered here, before
# the server is started, from the bytes that will be shipped.
import copy

from transformers import AutoTokenizer

probe_tokenizer = AutoTokenizer.from_pretrained('/kaggle/working/runs/atra-4b')
probe_tokenizer.chat_template = GGUF_TEMPLATE

messages = [
    {'role': 'system', 'content': 'system prompt'},
    {'role': 'user', 'content': 'user turn'},
]
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'get_market_snapshot',
            'description': 'Current price, liquidity and volume for a pool.',
            'parameters': {'type': 'object', 'required': ['chain'], 'properties': {'chain': {'type': 'string'}}},
        },
    }
]
plain = probe_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
with_tools = probe_tokenizer.apply_chat_template(
    messages, tools=tools, tokenize=False, add_generation_prompt=True
)
print()
print('rendered by the GGUF template, without tools:', len(plain), 'chars')
print('rendered by the GGUF template, with tools   :', len(with_tools), 'chars')
print('<tools> block present                       :', '<tools>' in with_tools)
print('tool name present                           :', 'get_market_snapshot' in with_tools)
assert 'get_market_snapshot' not in plain
assert 'get_market_snapshot' in with_tools, (
    'the template inside the GGUF ignores `tools`; serving it would repeat the '
    'first failure'
)
print()
print(with_tools[: len(with_tools) - len(plain) + 240])

pathlib.Path('/kaggle/working/gguf-chat-template.jinja').write_text(GGUF_TEMPLATE, encoding='utf-8')
del reader

## 7. The test set

Rebuilt deterministically, with the seed and per-domain count the thresholds
were written for, rather than carried: the build is the specification. The
digest is then compared with the working tree's copy, so "the same 95
examples" is checked rather than assumed.

In [ ]:
%%bash
set -e
set -o pipefail
cd /kaggle/working/atra
# The same command, seed and per-domain count the thresholds were written for.
python -m data.build --seed 42 --per-domain 200 --out data/out
python -m data.checks data/out > /kaggle/working/data-checks.txt 2>&1 || true
tail -3 /kaggle/working/data-checks.txt
echo
wc -l data/out/*.jsonl
echo
echo "=== the 95 examples this run is scored on ==="
sha256sum data/out/test.jsonl data/out/validation.jsonl data/out/train.jsonl

In [ ]:
import collections, hashlib, json, pathlib

# The test split rebuilt in-kernel has to be the same 95 examples the working
# tree holds, or "against the repo's thresholds" means nothing. The expected
# digest is the working tree's data/out/test.jsonl at notebook generation time.
EXPECTED_TEST_SHA = "6f870014d7608541f62753b3d4f66d9e99d3618576705b03e29809e705ffc229"

TEST = pathlib.Path('/kaggle/working/atra/data/out/test.jsonl')
raw = TEST.read_bytes()
digest = hashlib.sha256(raw).hexdigest()
print('test.jsonl in-kernel   :', digest)
print('test.jsonl working tree:', EXPECTED_TEST_SHA)
print('identical              :', digest == EXPECTED_TEST_SHA)
assert digest == EXPECTED_TEST_SHA, (
    'the deterministic rebuild does not reproduce the working tree split; '
    'the thresholds and the examples are no longer the same pair'
)

examples = [json.loads(line) for line in raw.decode('utf-8').splitlines() if line.strip()]
print()
print('examples   :', len(examples))
print('by domain  :', dict(collections.Counter(e['domain'] for e in examples)))
print('with tools :', sum(1 for e in examples if e.get('tools')))

## 8. Serve the GGUF — with nothing but `--jinja`

No `--chat-template-file`. The template being used has to be the one inside
the file, or the measurement describes a server configuration nobody else
will have.

In [ ]:
import hashlib, json, os, pathlib, subprocess, time, urllib.request

BIN = '/kaggle/temp/llama.cpp/build/bin/llama-server'
GGUF_PATH = '/kaggle/working/atra-4b-q4_k_m.gguf'
PORT = 8080
ENDPOINT = f'http://127.0.0.1:{PORT}'
LOG = pathlib.Path('/kaggle/working/llama-server.log')
BUILD_KIND = pathlib.Path('/kaggle/temp/build-kind.txt').read_text().strip()
print('build kind:', BUILD_KIND)

# No --chat-template-file this time, deliberately.
#
# The previous run had to hand llama-server a template on the command line
# because the GGUF carried none. That patched the measurement, not the
# artifact: anyone downloading the file would still have been served a default
# template with no tools block. This run serves the GGUF with nothing but
# --jinja, so the template being used is the one inside the file.
command = [
    BIN, '-m', GGUF_PATH,
    '--host', '127.0.0.1', '--port', str(PORT),
    '-c', '8192', '-ngl', '99', '--parallel', '1',
    '-t', str(os.cpu_count() or 4),
    '--jinja',
]
print('$', ' '.join(command))
handle = LOG.open('ab')
server = subprocess.Popen(command, stdout=handle, stderr=subprocess.STDOUT)


def wait_ready(timeout=1200):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if server.poll() is not None:
            print('server exited with', server.returncode)
            return False
        try:
            with urllib.request.urlopen(ENDPOINT + '/health', timeout=5) as response:
                if response.status == 200:
                    return True
        except Exception:
            pass
        time.sleep(3)
    return False


ready = wait_ready()
if not ready:
    print(LOG.read_text(errors='replace')[-6000:])
assert ready, 'llama-server never answered /health while serving the GGUF with --jinja alone'
print('server ready')

# What template is the server actually using? Its own answer, compared with
# the bytes read out of the GGUF two cells ago.
with urllib.request.urlopen(ENDPOINT + '/props', timeout=30) as response:
    props = json.loads(response.read())
served = props.get('chat_template') or (props.get('default_generation_settings') or {}).get('chat_template')
print()
print('/props keys        :', sorted(props.keys()))
if served:
    print('served template    :', len(served), 'chars')
    print('served sha256      :', hashlib.sha256(served.encode('utf-8')).hexdigest())
    print('GGUF sha256        :', GGUF_TEMPLATE_SHA)
    print('same as the GGUF   :', hashlib.sha256(served.encode('utf-8')).hexdigest() == GGUF_TEMPLATE_SHA)
    print('honours tools      :', '<tools>' in served or 'tools' in served)
else:
    print('/props does not report a chat template on this build; the prompt-token '
          'counts below are the evidence instead')
print()
print(json.dumps({k: v for k, v in props.items() if k != 'chat_template'}, indent=1)[:1500])
print()
print(LOG.read_text(errors='replace')[-1800:])

## 9. What the served model actually says

The prompt-shape check first: the same prompt sent with and without the tool
block, so the token counts show whether the schemas survive the trip. 91
tokens either way was the signature of failure #1. Then the same example
through `/completion` with the prompt rendered locally, and one example per
domain printed in full beside the expected answer.

In [ ]:
import json, pathlib, sys, time, urllib.request

sys.path.insert(0, '/kaggle/working/atra')

TEST = pathlib.Path('/kaggle/working/atra/data/out/test.jsonl')
examples = [json.loads(line) for line in TEST.read_text(encoding='utf-8').splitlines() if line.strip()]
print('test examples:', len(examples))


def tools_for(example):
    """The tool block, in the exact shape prompting.tool_schemas builds."""
    return [
        {
            'type': 'function',
            'function': {
                'name': tool['name'],
                'description': tool['description'],
                'parameters': tool['parameters'],
            },
        }
        for tool in example.get('tools', [])
    ]


def prompt_of(example):
    return [
        {'role': message['role'], 'content': message['content']}
        for message in example['messages'] if message['role'] != 'assistant'
    ]


def post(path, body, timeout=900):
    request = urllib.request.Request(
        ENDPOINT + path,
        data=json.dumps(body).encode('utf-8'),
        headers={'content-type': 'application/json'},
    )
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read())


def ask_chat(messages, tools=None):
    body = {
        'model': 'atra-4b-v0',
        'messages': messages,
        'temperature': 0.0,
        'max_tokens': 512,
        'stream': False,
    }
    if tools:
        body['tools'] = tools
    return post('/v1/chat/completions', body)


# ---------------------------------------------------------------------------
# 1. Does the tool block reach the model through the server's own template?
#
# 91 prompt tokens with and without `tools` was the signature of the second
# failure: evaluate.py sent the schemas and llama-server dropped them, because
# the GGUF had no template. Same request, both ways, before ~95 generations.
# ---------------------------------------------------------------------------
tool_example = next(e for e in examples if e['domain'] == 'tool_use')
with_tools = ask_chat(prompt_of(tool_example), tools_for(tool_example))
without_tools = ask_chat(prompt_of(tool_example), None)

n_with = with_tools.get('usage', {}).get('prompt_tokens')
n_without = without_tools.get('usage', {}).get('prompt_tokens')
message = with_tools['choices'][0]['message']

print('=' * 78)
print('PROMPT SHAPE CHECK  (example:', tool_example['id'], ')')
print('  prompt_tokens WITHOUT tools :', n_without, '  <- what the first, broken run sent')
print('  prompt_tokens WITH tools    :', n_with, '  <- what the fixed harness sends')
print('  difference                  :', (n_with - n_without) if (n_with and n_without) else '?')
print('  tools in the example        :', [t['name'] for t in tool_example.get('tools', [])])
print('  reply message keys          :', sorted(message.keys()))
print('  content is empty            :', not (message.get('content') or '').strip())
print('  tool_calls present          :', bool(message.get('tool_calls')))
print('  finish_reason               :', with_tools['choices'][0].get('finish_reason'))
print('  raw message                 :', json.dumps(message, sort_keys=True)[:1200])
print('=' * 78)

assert n_with and n_without and n_with > n_without, (
    f'the tool block did not change the prompt length ({n_without} -> {n_with}); '
    'the server is not rendering tools and the run would repeat the original bug'
)
# Not an assertion any more. evaluate.py now rebuilds the reply from
# `tool_calls` when llama.cpp's parser moves it there, and records that it did
# so in the trace. Whether the parser fires is reported, not fatal.
if not (message.get('content') or '').strip() and message.get('tool_calls'):
    print()
    print("NOTE: llama.cpp's tool-call parser moved this reply out of content and")
    print('      into tool_calls. evaluate.py reconstructs ATRA\'s reply shape from')
    print('      it (EndpointModel._from_tool_calls) and flags each such reply in')
    print('      the trace as reconstructed_from_tool_calls.')

# ---------------------------------------------------------------------------
# 2. The other serving path on the same server: /completion with the prompt
#    rendered here by prompting.render_example — the same function train.py
#    used. Nothing renders, nothing parses. This is evaluate.py's default.
# ---------------------------------------------------------------------------
from data.checks import load_jsonl
from evaluate import load_tokenizer
from prompting import render_example

tokenizer = load_tokenizer('/kaggle/working/runs/atra-4b')
sample = next(e for e in load_jsonl(TEST) if e.id == tool_example['id'])

rendered = render_example(sample, tokenizer, prompt_only=True)
local_tokens = len(tokenizer(rendered, add_special_tokens=False)['input_ids'])
raw = post('/completion', {
    'prompt': rendered,
    'temperature': 0.0,
    'top_k': 1,
    'n_predict': 512,
    'cache_prompt': True,
    'stream': False,
    'stop': ['<|im_end|>', '<|endoftext|>'],
})
print()
print('=' * 78)
print('RAW /completion CHECK  (prompt rendered here, by the training template)')
print('  prompt chars                :', len(rendered))
print('  prompt tokens, rendered here:', local_tokens)
print('  prompt tokens, per server   :', raw.get('tokens_evaluated'))
print('  <tools> in the prompt       :', '<tools>' in rendered)
print('  tokens generated            :', raw.get('tokens_predicted'))
print('  stop type                   :', raw.get('stop_type'))
print('  reply                       :', (raw.get('content') or '')[:900])
print('=' * 78)

# ---------------------------------------------------------------------------
# 3. One example per domain, printed in full next to the expected answer. If a
#    metric fails later, this is the evidence for why.
# ---------------------------------------------------------------------------
seen = set()
probes = []
for example in examples:
    if example['domain'] not in seen:
        seen.add(example['domain'])
        probes.append(example)

for example in probes:
    started = time.time()
    body = ask_chat(prompt_of(example), tools_for(example))
    elapsed = time.time() - started
    reply = body['choices'][0]['message']
    print('=' * 78)
    print('id       :', example['id'], '| domain:', example['domain'], '| tags:', example.get('tags'))
    print('user     :', prompt_of(example)[-1]['content'][:300])
    print('expected :', json.dumps(example['expected_output'], sort_keys=True)[:400])
    print('reply    :', (reply.get('content') or '')[:900])
    if reply.get('tool_calls'):
        print('tool_call:', json.dumps(reply['tool_calls'], sort_keys=True)[:600])
    print(f'timing   : {elapsed:.1f}s  usage={body.get("usage", {})}')

## 10. Evaluate — both serving paths, one server, one split

In [ ]:
%%bash
# Deliberately no `set -e`: evaluate.py exits non-zero when a threshold is
# missed (1) or when the serving path ate the replies (3), and both are
# results to report rather than kernel failures.
set -o pipefail
cd /kaggle/working/atra

stat -c %s /kaggle/working/llama-server.log > /kaggle/working/log-offset-chat.txt

echo "################################################################"
echo "# PASS A  --serving chat"
echo "#   POST /v1/chat/completions with the tools; llama-server renders"
echo "#   the GGUF's own template and runs its tool-call parser. This is"
echo "#   how the ATRA runtime and Ollama talk to the model."
echo "################################################################"
python evaluate.py \
  --data data/out/test.jsonl \
  --model http://127.0.0.1:8080 \
  --model-name atra-4b-v0 \
  --serving chat \
  --tokenizer /kaggle/working/runs/atra-4b \
  --trace /kaggle/working/trace-chat.jsonl \
  --out /kaggle/working/eval-chat.json \
  --config config/default.yaml 2>&1 | tee /kaggle/working/evaluate-chat-stdout.txt
CHAT=${PIPESTATUS[0]}
echo "${CHAT}" > /kaggle/working/evaluate-chat-exit-code.txt
echo
echo "evaluate.py --serving chat exit code: ${CHAT}"

stat -c %s /kaggle/working/llama-server.log > /kaggle/working/log-offset-raw.txt

echo
echo "################################################################"
echo "# PASS B  --serving raw   (evaluate.py's default)"
echo "#   POST /completion with the prompt rendered here by"
echo "#   prompting.render_example — the same function train.py used."
echo "#   Nothing renders server-side, nothing parses the reply."
echo "################################################################"
python evaluate.py \
  --data data/out/test.jsonl \
  --model http://127.0.0.1:8080 \
  --model-name atra-4b-v0 \
  --serving raw \
  --tokenizer /kaggle/working/runs/atra-4b \
  --trace /kaggle/working/trace-raw.jsonl \
  --out /kaggle/working/eval-raw.json \
  --config config/default.yaml 2>&1 | tee /kaggle/working/evaluate-raw-stdout.txt
RAW=${PIPESTATUS[0]}
echo "${RAW}" > /kaggle/working/evaluate-raw-exit-code.txt
echo
echo "evaluate.py --serving raw  exit code: ${RAW}"
echo
echo "exit codes: chat=${CHAT} raw=${RAW}   (0 pass, 1 threshold missed, 3 replies lost content)"

## 11. Did the tools reach the model, and did the replies reach the scorer?

From two independent records: the per-reply trace `evaluate.py` wrote, and
llama-server's own log. `replies_losing_content` is the number that decides
whether any of the metrics below may be quoted at all.

In [ ]:
import json, pathlib, re, statistics

# Direct evidence that the tool block reached the model on every scored
# request, not just on the probes — from two independent records: the trace
# evaluate.py wrote per reply, and llama-server's own log.
LOG = pathlib.Path('/kaggle/working/llama-server.log')
log_text = LOG.read_text(errors='replace')


def from_trace(path):
    rows = [json.loads(line) for line in pathlib.Path(path).read_text().splitlines() if line.strip()]
    return rows


def describe(name, rows):
    print('=' * 78)
    print(name)
    print('  replies                     :', len(rows))
    for key, label in (
        ('prompt_tokens_server', 'prompt tokens, per the server'),
        ('prompt_tokens_local', 'prompt tokens, rendered here'),
        ('generated_tokens', 'tokens generated'),
        ('returned_tokens', 'tokens in the returned text'),
    ):
        values = [r[key] for r in rows if r.get(key) is not None]
        if values:
            print(f'  {label:<28}: min {min(values)}  median {int(statistics.median(values))}  '
                  f'max {max(values)}  mean {statistics.fmean(values):.1f}')
        else:
            print(f'  {label:<28}: not recorded by this path')
    short = [r for r in rows if (r.get('prompt_tokens_server') or 0) and r['prompt_tokens_server'] < 200]
    print('  prompts under 200 tokens    :', len(short),
          '  <- the first, broken run sat at 91-119 on every request')
    empty = [r for r in rows if not r.get('reply_chars')]
    print('  empty replies               :', len(empty), [r['id'] for r in empty[:10]])
    rebuilt = [r for r in rows if r.get('reconstructed_from_tool_calls')]
    print('  rebuilt from tool_calls     :', len(rebuilt), [r['id'] for r in rebuilt[:10]])
    trunc = [r for r in rows if r.get('finish_reason') == 'length' or r.get('truncated')]
    print('  hit the 512-token ceiling   :', len(trunc), [r['id'] for r in trunc[:10]])
    lost = [
        r for r in rows
        if r.get('generated_tokens') is not None
        and r.get('returned_tokens') is not None
        and r['returned_tokens'] < r['generated_tokens'] - 2
    ]
    print('  REPLIES LOSING CONTENT      :', len(lost))
    for r in lost[:10]:
        print(f"      {r['id']}: server generated {r['generated_tokens']}, "
              f"text carries {r['returned_tokens']}")
    print()


for label, path in (
    ('PASS A  --serving chat  (/v1/chat/completions)', '/kaggle/working/trace-chat.jsonl'),
    ('PASS B  --serving raw   (/completion)', '/kaggle/working/trace-raw.jsonl'),
):
    if pathlib.Path(path).exists():
        describe(label, from_trace(path))
    else:
        print(label, '-> no trace written')

# llama-server's own record, split at the byte offset taken before each pass.
print('=' * 78)
print("llama-server's own prompt lengths")
offsets = {}
for key in ('chat', 'raw'):
    p = pathlib.Path(f'/kaggle/working/log-offset-{key}.txt')
    offsets[key] = int(p.read_text().strip()) if p.exists() else 0

for key, start, end in (
    ('PASS A chat', offsets['chat'], offsets['raw'] or len(log_text)),
    ('PASS B raw ', offsets['raw'], len(log_text)),
):
    segment = log_text[start:end]
    counts = [int(n) for n in re.findall(r'prompt eval time.*?/\s*(\d+)\s*tokens', segment)]
    if not counts:
        counts = [int(n) for n in re.findall(r'n_prompt_tokens\s*=\s*(\d+)', segment)]
    if counts:
        print(f'  {key}: {len(counts)} prompt evals, min {min(counts)}, '
              f'median {int(statistics.median(counts))}, max {max(counts)}')
    else:
        print(f'  {key}: no prompt-eval lines in this segment of the log')

## 12. Result

In [ ]:
import json, pathlib

EXIT_MEANING = {
    '0': 'PASS — every threshold met',
    '1': 'FAIL — at least one metric below threshold',
    '2': 'ERROR — the harness could not run',
    '3': 'CONTAMINATED — replies lost content between the model and the scorer',
}


def report(title, eval_path, code_path):
    path = pathlib.Path(eval_path)
    if not path.exists():
        print(title, '-> no eval.json; this pass did not finish')
        return None
    data = json.loads(path.read_text(encoding='utf-8'))
    code = pathlib.Path(code_path).read_text().strip()

    print('=' * 104)
    print(title)
    print('=' * 104)
    print('model        :', data['model'])
    print('backend      :', data.get('backend'))
    print('dataset hash :', data['dataset_hash'])
    print('examples     :', data['examples'])
    print('ran at       :', data['ran_at'], f"({data['duration_sec']}s)")
    print('gguf sha256  :', GGUF_SHA)
    print('gguf bytes   :', GGUF_BYTES)
    print()

    header = (f"{'metric':<30}{'score':>9}{'threshold':>12}{'direction':>17}"
              f"{'passed':>12}{'verdict':>10}{'margin':>12}")
    print(header)
    print('-' * len(header))
    missed = []
    for metric in data['metrics']:
        threshold = metric['threshold']
        lower = metric['lower_is_better']
        direction = 'lower is better' if lower else 'higher is better'
        fraction = f"{metric['passed']}/{metric['total']}"
        if threshold is None:
            verdict, margin, shown = 'n/a', '', 'n/a'
        else:
            ok = metric['meets_threshold']
            verdict = 'PASS' if ok else 'FAIL'
            delta = (threshold - metric['score']) if lower else (metric['score'] - threshold)
            margin = f'{delta:+.4f}'
            shown = f'{threshold:.4f}'
            if not ok:
                missed.append((metric['name'], metric['score'], threshold, delta, lower))
        print(f"{metric['name']:<30}{metric['score']:>9.4f}{shown:>12}{direction:>17}"
              f"{fraction:>12}{verdict:>10}{margin:>12}")

    print()
    if missed:
        print(f'{len(missed)} metric(s) BELOW THRESHOLD:')
        for name, score, threshold, delta, lower in missed:
            word = 'above the maximum by' if lower else 'short of the minimum by'
            print(f'  - {name}: {score:.4f} vs {threshold:.4f} ({word} {abs(delta):.4f})')
    else:
        print('every metric meets its threshold.')

    accounting = data.get('content_accounting') or {}
    print()
    print('content accounting')
    print('  replies                     :', accounting.get('replies'))
    print('  checked                     :', accounting.get('checked'))
    print('  generated tokens (server)   :', accounting.get('generated_tokens_total'))
    print('  tokens in the returned text :', accounting.get('returned_tokens_total'))
    print('  empty replies               :', accounting.get('empty_replies'))
    print('  REPLIES_LOSING_CONTENT      :', accounting.get('replies_losing_content'))
    print('  prompt tokens, rendered here:',
          accounting.get('prompt_tokens_local_min'), '-', accounting.get('prompt_tokens_local_max'))
    print('  prompt tokens, per server   :',
          accounting.get('prompt_tokens_server_min'), '-', accounting.get('prompt_tokens_server_max'))
    print('  local/server disagreements  :', accounting.get('prompt_tokens_local_vs_server_mismatch'))
    for row in accounting.get('worst', []) or []:
        print(f"      {row['id']}: generated {row['generated_tokens']}, returned {row['returned_tokens']}")

    print()
    print('example ids that failed each metric (evaluate.py records the first ten):')
    for metric in data['metrics']:
        if metric['failures']:
            print(f"  {metric['name']}: {metric['failures']}")

    print()
    print('evaluate.py exit code:', code, '->', EXIT_MEANING.get(code, 'unknown'))
    print()
    return code, data


chat = report('PASS A — q4_k_m GGUF via llama-server /v1/chat/completions (tools sent, server renders and parses)',
              '/kaggle/working/eval-chat.json', '/kaggle/working/evaluate-chat-exit-code.txt')
raw = report('PASS B — q4_k_m GGUF via llama-server /completion (prompt rendered by prompting.render_example)',
             '/kaggle/working/eval-raw.json', '/kaggle/working/evaluate-raw-exit-code.txt')

print('=' * 104)
print('SUMMARY')
print('=' * 104)
print('gguf sha256 :', GGUF_SHA)
print('gguf bytes  :', GGUF_BYTES)
print('build kind  :', BUILD_KIND)
for name, result in (('chat', chat), ('raw', raw)):
    if result is None:
        print(f'{name:<5}: did not finish')
        continue
    code, data = result
    losing = (data.get('content_accounting') or {}).get('replies_losing_content')
    print(f'{name:<5}: exit {code} ({EXIT_MEANING.get(code, "unknown")}), '
          f'replies_losing_content={losing}')

print()
print('=' * 104)
print('raw eval-chat.json')
print('=' * 104)
print(pathlib.Path('/kaggle/working/eval-chat.json').read_text()
      if pathlib.Path('/kaggle/working/eval-chat.json').exists() else 'missing')
print()
print('=' * 104)
print('raw eval-raw.json')
print('=' * 104)
print(pathlib.Path('/kaggle/working/eval-raw.json').read_text()
      if pathlib.Path('/kaggle/working/eval-raw.json').exists() else 'missing')

## 13. What is kept

- `eval-chat.json`, `eval-raw.json` — the two results, thresholds included
- `trace-chat.jsonl`, `trace-raw.jsonl` — every reply with its token accounting
- `atra-4b-q4_k_m.gguf` + `Modelfile` — the exact artifact that was evaluated
- `export-manifest.json`, `export.log` — adapter, base revision, training manifest
- `gguf-chat-template.jinja` — the template read back out of the GGUF
- `evaluate-*-stdout.txt`, `evaluate-*-exit-code.txt`, `data-checks.txt`

Whether the repository's UNTRAINED label changes is decided by the numbers
above, and not in this notebook.

In [ ]:
%%bash
set +e
pkill -f llama-server
sleep 3
# The server log is useful but not an artifact; keep only its tail.
tail -c 400000 /kaggle/working/llama-server.log > /kaggle/working/llama-server.tail.log 2>/dev/null
rm -f /kaggle/working/llama-server.log
rm -rf /kaggle/working/runs /kaggle/working/atra/data/out/train.jsonl
ls -la /kaggle/working
du -sh /kaggle/working
echo "done"